# Order Balance: CrewAI 4-Agent 프랜차이즈 발주 검토

판매·재고·발주·예산 Agent 전용 CSV를 Python으로 검증하고, 4개의 LLM Agent가 순차적으로 결과를 해석하는 교육용 노트북입니다.

**실행 순서:** 판매분석가 → 재고관리자 → 발주담당자 → 예산관리자  
**역할 분리:** Python은 수치 계산·제약 검증, LLM은 해석·설명·검토를 담당합니다.


## 1. 라이브러리 및 환경 설정


In [ ]:
# 최초 1회만 실행하세요.
# %pip install -U "crewai[openai]>=1.15,<2.0" python-dotenv pandas numpy reportlab


In [ ]:
import json
import math
import os
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from dotenv import load_dotenv

# CrewAI SQLite 상태 파일을 현재 프로젝트의 쓰기 가능한 폴더에 저장합니다.
CREWAI_STORAGE_PATH = Path.cwd() / ".crewai_storage"
CREWAI_STORAGE_PATH.mkdir(parents=True, exist_ok=True)
os.environ["CREWAI_STORAGE_DIR"] = str(CREWAI_STORAGE_PATH)

from crewai import Agent, Crew, LLM, Process, Task
from crewai.tools import tool

env_candidates = [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
    Path.cwd().parent.parent / ".env",
]
env_path = next((path for path in env_candidates if path.exists()), None)
if env_path is not None:
    load_dotenv(dotenv_path=env_path, override=False)
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 OPENAI_API_KEY를 설정하세요.")

model_name = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini")
if not model_name.startswith("openai/"):
    model_name = f"openai/{model_name}"

def make_agent_llm():
    return LLM(
        model=model_name,
        api_key=api_key,
        temperature=0.2,
    )


# Agent별 인스턴스를 분리해야 CrewAI 토큰 사용량이 중복 합산되지 않습니다.
sales_llm = make_agent_llm()
inventory_llm = make_agent_llm()
ordering_llm = make_agent_llm()
budget_llm = make_agent_llm()
print(f"사용 모델: {model_name}")


## 2. CSV 경로 설정


In [ ]:
BASE_DIR = Path.cwd().resolve()
CANDIDATE_DIRS = [
    BASE_DIR,
    BASE_DIR / "data",
    BASE_DIR / "csv",
    BASE_DIR / "LLM" / "code2",
    BASE_DIR / "LLM" / "code2" / "data",
]


def normalized_name(name):
    return unicodedata.normalize("NFC", str(name))


def find_csv_file(logical_filename, required=True):
    target_name = normalized_name(logical_filename)
    searched = []
    for directory in CANDIDATE_DIRS:
        if not directory.is_dir():
            continue
        searched.append(str(directory))
        for path in directory.iterdir():
            if path.is_file() and normalized_name(path.name) == target_name:
                return path
    if required:
        raise FileNotFoundError(
            f"{logical_filename} 파일을 찾을 수 없습니다. 검색 위치: {searched}"
        )
    return None


DATA_DIR = find_csv_file("agent_재고관리자.csv").parent
OUTPUT_DIR = DATA_DIR.parent if DATA_DIR.name in {"data", "csv"} else DATA_DIR
print(f"입력 데이터 폴더: {DATA_DIR}")
print(f"출력 폴더: {OUTPUT_DIR}")


## 3. 판매 입력 모드 선택


In [ ]:
# 아래 두 줄 중 하나만 활성화하세요.
SALES_INPUT_MODE = "SUMMARY"
# SALES_INPUT_MODE = "DAILY_DETAIL"

STORE_ID = "ST001"
NUMERIC_TOLERANCE = 0.02

if SALES_INPUT_MODE not in {"SUMMARY", "DAILY_DETAIL"}:
    raise ValueError("SALES_INPUT_MODE는 SUMMARY 또는 DAILY_DETAIL이어야 합니다.")


## 4. 공통 데이터 변환 함수


In [ ]:
TRUE_VALUES = {"TRUE", "1", "Y", "YES"}
FALSE_VALUES = {"FALSE", "0", "N", "NO"}


def to_bool_value(value):
    if pd.isna(value):
        return False
    normalized = str(value).strip().upper()
    if normalized in TRUE_VALUES:
        return True
    if normalized in FALSE_VALUES:
        return False
    raise ValueError(f"bool로 변환할 수 없는 값: {value}")


def require_columns(df, required_columns, source_name):
    missing = sorted(set(required_columns) - set(df.columns))
    if missing:
        raise ValueError(f"{source_name} 필수 컬럼 누락: {missing}")


def convert_columns(df, date_columns=(), bool_columns=(), numeric_columns=()):
    result = df.copy()
    for column in date_columns:
        if column in result:
            result[column] = pd.to_datetime(result[column], errors="raise")
    for column in bool_columns:
        if column in result:
            result[column] = result[column].map(to_bool_value).astype(bool)
    for column in numeric_columns:
        if column in result:
            result[column] = pd.to_numeric(result[column], errors="raise")
    return result


def validate_unique(df, key_columns, source_name):
    duplicates = df[df.duplicated(list(key_columns), keep=False)]
    if not duplicates.empty:
        raise ValueError(
            f"{source_name} 키 중복: {list(key_columns)}, {len(duplicates)}행"
        )


def load_typed_agent_csv(
    path,
    required_columns,
    date_columns=(),
    bool_columns=(),
    numeric_columns=(),
    key_columns=("store_id", "item_id"),
):
    df = pd.read_csv(path, encoding="utf-8-sig")
    require_columns(df, required_columns, path.name)
    df = convert_columns(df, date_columns, bool_columns, numeric_columns)
    df = df[df["store_id"].astype(str) == STORE_ID].copy()
    validate_unique(df, key_columns, path.name)
    return df.sort_values(list(key_columns)).reset_index(drop=True)


def compact_json(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        separators=(",", ":"),
        default=str,
    )


def json_records(df, columns):
    safe = df.loc[:, columns].copy()
    safe = safe.astype(object).where(pd.notna(safe), None)
    return safe.to_dict(orient="records")


## 5. SUMMARY 판매 입력


In [ ]:
SALES_SUMMARY_REQUIRED = [
    "as_of_date", "store_id", "item_id", "gtin", "item_name",
    "category_l1", "category_l2", "is_perishable",
    "sales_short_window_days", "sales_long_window_days",
    "sales_quantity_short", "sales_quantity_long",
    "avg_daily_quantity_short", "avg_daily_quantity_long",
    "std_daily_quantity_long", "cv_daily_quantity_long",
    "recent_to_long_ratio", "sales_trend_qty_per_day_long",
    "weekday_avg_quantity", "weekend_avg_quantity", "weekend_uplift_pct",
    "top_sales_weekday", "top_sales_weekday_avg_quantity",
    "promotion_sales_quantity_long", "promotion_quantity_share_pct",
    "zero_sales_days_long",
]
SALES_SUMMARY_NUMERIC = SALES_SUMMARY_REQUIRED[8:21] + [
    "top_sales_weekday_avg_quantity",
    "promotion_sales_quantity_long",
    "promotion_quantity_share_pct",
    "zero_sales_days_long",
]


def load_sales_summary_csv(path):
    df = load_typed_agent_csv(
        path,
        SALES_SUMMARY_REQUIRED,
        date_columns=["as_of_date"],
        bool_columns=["is_perishable"],
        numeric_columns=SALES_SUMMARY_NUMERIC,
    )
    df["sales_input_mode"] = "SUMMARY"
    return df


## 6. DAILY_DETAIL 판매 입력


In [ ]:
SALES_DAILY_REQUIRED = [
    "business_date", "store_id", "item_id", "gtin", "item_name",
    "category_l1", "category_l2", "is_perishable", "day_of_week",
    "weekend_yn", "sales_quantity", "gross_sales_amount_krw",
    "discount_amount_krw", "net_sales_amount_krw", "transaction_count",
    "sales_line_count", "promotion_sales_quantity", "promotion_line_count",
]
DAY_ORDER = [
    "MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY",
    "FRIDAY", "SATURDAY", "SUNDAY",
]


def build_sales_summary_from_daily(path):
    daily = pd.read_csv(path, encoding="utf-8-sig")
    require_columns(daily, SALES_DAILY_REQUIRED, path.name)
    daily = convert_columns(
        daily,
        date_columns=["business_date"],
        bool_columns=["is_perishable", "weekend_yn"],
        numeric_columns=[
            "sales_quantity", "gross_sales_amount_krw",
            "discount_amount_krw", "net_sales_amount_krw",
            "transaction_count", "sales_line_count",
            "promotion_sales_quantity", "promotion_line_count",
        ],
    )
    daily = daily[daily["store_id"].astype(str) == STORE_ID].copy()
    validate_unique(daily, ["business_date", "store_id", "item_id"], path.name)
    as_of_date = daily["business_date"].max()
    short_start = as_of_date - pd.Timedelta(days=6)
    long_start = as_of_date - pd.Timedelta(days=27)
    rows = []

    for (store_id, item_id), group in daily.groupby(["store_id", "item_id"], sort=True):
        long_df = group[
            group["business_date"].between(long_start, as_of_date)
        ].sort_values("business_date")
        short_df = long_df[long_df["business_date"] >= short_start]
        if len(short_df) != 7 or len(long_df) != 28:
            raise ValueError(
                f"{item_id}: 최근 7일/28일 행 수가 7/28이 아닙니다: "
                f"{len(short_df)}/{len(long_df)}"
            )

        first = group.iloc[0]
        short_avg = short_df["sales_quantity"].mean()
        long_avg = long_df["sales_quantity"].mean()
        std_long = long_df["sales_quantity"].std(ddof=0)
        weekday_avg = long_df.loc[
            ~long_df["weekend_yn"], "sales_quantity"
        ].mean()
        weekend_avg = long_df.loc[
            long_df["weekend_yn"], "sales_quantity"
        ].mean()
        weekday_means = (
            long_df.groupby("day_of_week")["sales_quantity"].mean().to_dict()
        )
        top_day = max(
            DAY_ORDER,
            key=lambda day: (weekday_means.get(day, float("-inf")), -DAY_ORDER.index(day)),
        )
        sales_long = long_df["sales_quantity"].sum()
        promo_long = long_df["promotion_sales_quantity"].sum()
        trend = np.polyfit(
            np.arange(28, dtype=float),
            long_df["sales_quantity"].to_numpy(dtype=float),
            1,
        )[0]

        rows.append({
            "as_of_date": as_of_date,
            "store_id": store_id,
            "item_id": item_id,
            "gtin": str(first["gtin"]),
            "item_name": first["item_name"],
            "category_l1": first["category_l1"],
            "category_l2": first["category_l2"],
            "is_perishable": bool(first["is_perishable"]),
            "sales_short_window_days": 7,
            "sales_long_window_days": 28,
            "sales_quantity_short": short_df["sales_quantity"].sum(),
            "sales_quantity_long": sales_long,
            "avg_daily_quantity_short": short_avg,
            "avg_daily_quantity_long": long_avg,
            "std_daily_quantity_long": std_long,
            "cv_daily_quantity_long": std_long / long_avg if long_avg else 0,
            "recent_to_long_ratio": short_avg / long_avg if long_avg else 0,
            "sales_trend_qty_per_day_long": trend,
            "weekday_avg_quantity": weekday_avg,
            "weekend_avg_quantity": weekend_avg,
            "weekend_uplift_pct": (
                (weekend_avg / weekday_avg - 1) * 100 if weekday_avg else 0
            ),
            "top_sales_weekday": top_day,
            "top_sales_weekday_avg_quantity": weekday_means.get(top_day, 0),
            "promotion_sales_quantity_long": promo_long,
            "promotion_quantity_share_pct": (
                promo_long / sales_long * 100 if sales_long else 0
            ),
            "zero_sales_days_long": int((long_df["sales_quantity"] == 0).sum()),
            "sales_input_mode": "DAILY_DETAIL",
        })

    result = pd.DataFrame(rows)
    numeric = [column for column in SALES_SUMMARY_NUMERIC if column in result]
    result[numeric] = result[numeric].round(2)
    validate_unique(result, ["store_id", "item_id"], "DAILY_DETAIL 요약")
    return result.sort_values(["store_id", "item_id"]).reset_index(drop=True)


## 7. 판매 모드 선택 및 비교 검증


In [ ]:
summary_path = find_csv_file("agent_판매분석가.csv")
daily_path = find_csv_file("agent_판매분석가_일별상세.csv")

if SALES_INPUT_MODE == "SUMMARY":
    sales_agent_df = load_sales_summary_csv(summary_path)
elif SALES_INPUT_MODE == "DAILY_DETAIL":
    sales_agent_df = build_sales_summary_from_daily(daily_path)
else:
    raise ValueError("SALES_INPUT_MODE는 SUMMARY 또는 DAILY_DETAIL이어야 합니다.")


SALES_COMPARE_COLUMNS = [
    "sales_quantity_short", "sales_quantity_long",
    "avg_daily_quantity_short", "avg_daily_quantity_long",
    "std_daily_quantity_long", "recent_to_long_ratio",
    "sales_trend_qty_per_day_long", "weekday_avg_quantity",
    "weekend_avg_quantity", "weekend_uplift_pct", "top_sales_weekday",
    "top_sales_weekday_avg_quantity", "promotion_sales_quantity_long",
    "promotion_quantity_share_pct", "zero_sales_days_long",
]


def compare_sales_summary_versions(display_result=True):
    summary = load_sales_summary_csv(summary_path)
    daily_summary = build_sales_summary_from_daily(daily_path)
    merged = summary.merge(
        daily_summary,
        on=["store_id", "item_id"],
        suffixes=("_summary", "_daily"),
        validate="one_to_one",
    )
    rows = []
    for _, row in merged.iterrows():
        for column in SALES_COMPARE_COLUMNS:
            left = row[f"{column}_summary"]
            right = row[f"{column}_daily"]
            if column == "top_sales_weekday":
                difference = "" if left == right else "문자열 불일치"
                status = "PASS" if left == right else "REVIEW_REQUIRED"
            else:
                difference = abs(float(left) - float(right))
                status = "PASS" if difference <= NUMERIC_TOLERANCE else "REVIEW_REQUIRED"
            rows.append({
                "item_id": row["item_id"],
                "컬럼": column,
                "SUMMARY 값": left,
                "DAILY_DETAIL 계산값": right,
                "차이": difference,
                "결과": status,
            })
    comparison = pd.DataFrame(rows)
    if display_result:
        print(comparison["결과"].value_counts().to_dict())
        display(comparison[comparison["결과"] != "PASS"].head(30))
    return comparison


sales_mode_comparison_df = compare_sales_summary_versions()
print(f"판매 입력 모드: {SALES_INPUT_MODE}")
print(f"판매 데이터 행 수: {len(sales_agent_df)}")
print(f"판매분석 대상 SKU 수: {sales_agent_df['item_id'].nunique()}")
display(sales_agent_df.head())


## 8. 재고·발주·예산 CSV 로딩


In [ ]:
INVENTORY_REQUIRED = [
    "snapshot_at", "snapshot_date", "store_id", "item_id", "item_name",
    "category_l1", "is_perishable", "sales_long_window_days",
    "avg_daily_quantity_long", "lead_time_days", "lead_time_demand_qty",
    "stockout_gap_qty", "on_hand_qty", "reserved_qty", "inbound_qty",
    "available_qty", "inventory_position_qty", "next_inbound_date",
    "inventory_cover_days", "lot_count", "nearest_expiry_date",
    "nearest_removal_date", "removal_within_short_horizon_qty",
    "removal_within_medium_horizon_qty", "removal_within_long_horizon_qty",
    "expiry_short_horizon_days", "expiry_medium_horizon_days",
    "expiry_long_horizon_days", "waste_risk_qty", "waste_risk_yn",
    "stockout_risk_yn", "stockout_warning_days", "stockout_warning_yn",
]
ORDERING_REQUIRED = [
    "snapshot_at", "store_id", "item_id", "supplier_id", "item_name",
    "category_l1", "supplier_name", "base_uom", "unit_cost_krw", "currency",
    "order_uom", "units_per_order_uom", "min_order_qty", "max_order_qty",
    "order_multiple_qty", "lead_time_days", "review_period_days",
    "safety_stock_days", "avg_daily_quantity_long", "std_daily_quantity_long",
    "available_qty", "inbound_qty", "inventory_position_qty",
    "inventory_cover_days", "lead_time_demand_qty", "stockout_gap_qty",
    "waste_risk_qty", "waste_risk_yn", "stockout_risk_yn",
    "stockout_warning_days", "stockout_warning_yn", "target_stock_qty",
    "net_required_qty", "recommended_order_qty_before_budget",
    "recommended_order_qty_before_budget_ea",
    "recommended_order_qty_before_budget_order_uom",
    "order_cap_applied_yn", "estimated_order_cost_krw",
]
BUDGET_REQUIRED = [
    "snapshot_at", "store_id", "item_id", "supplier_id", "item_name",
    "category_l1", "base_uom", "order_uom", "units_per_order_uom",
    "min_order_qty", "max_order_qty", "order_multiple_qty", "lead_time_days",
    "lead_time_demand_qty", "stockout_gap_qty", "stockout_risk_yn",
    "stockout_warning_yn", "waste_risk_yn", "inventory_cover_days",
    "waste_risk_qty", "service_level_target_pct",
    "recommended_order_qty_before_budget", "unit_cost_krw",
    "estimated_order_cost_krw", "total_candidate_cost_krw",
    "budget_limit_krw", "available_budget_krw",
    "candidate_cost_share_pct", "budget_shortfall_krw",
    "budget_remaining_krw", "budget_exceeded_yn",
]


def load_inventory_agent_csv(path):
    return load_typed_agent_csv(
        path, INVENTORY_REQUIRED,
        date_columns=[
            "snapshot_at", "snapshot_date", "next_inbound_date",
            "nearest_expiry_date", "nearest_removal_date",
        ],
        bool_columns=[
            "is_perishable", "waste_risk_yn",
            "stockout_risk_yn", "stockout_warning_yn",
        ],
        numeric_columns=[
            column for column in INVENTORY_REQUIRED
            if column not in {
                "snapshot_at", "snapshot_date", "store_id", "item_id",
                "item_name", "category_l1", "is_perishable",
                "next_inbound_date", "nearest_expiry_date",
                "nearest_removal_date", "waste_risk_yn",
                "stockout_risk_yn", "stockout_warning_yn",
            }
        ],
    )


def load_ordering_agent_csv(path):
    return load_typed_agent_csv(
        path, ORDERING_REQUIRED,
        date_columns=["snapshot_at"],
        bool_columns=[
            "waste_risk_yn", "stockout_risk_yn",
            "stockout_warning_yn", "order_cap_applied_yn",
        ],
        numeric_columns=[
            column for column in ORDERING_REQUIRED
            if column not in {
                "snapshot_at", "store_id", "item_id", "supplier_id",
                "item_name", "category_l1", "supplier_name", "base_uom",
                "currency", "order_uom", "waste_risk_yn",
                "stockout_risk_yn", "stockout_warning_yn",
                "order_cap_applied_yn",
            }
        ],
    )


def load_budget_agent_csv(path):
    return load_typed_agent_csv(
        path, BUDGET_REQUIRED,
        date_columns=["snapshot_at"],
        bool_columns=[
            "stockout_risk_yn", "stockout_warning_yn",
            "waste_risk_yn", "budget_exceeded_yn",
        ],
        numeric_columns=[
            column for column in BUDGET_REQUIRED
            if column not in {
                "snapshot_at", "store_id", "item_id", "supplier_id",
                "item_name", "category_l1", "base_uom", "order_uom",
                "stockout_risk_yn", "stockout_warning_yn",
                "waste_risk_yn", "budget_exceeded_yn",
            }
        ],
    )


inventory_agent_df = load_inventory_agent_csv(
    find_csv_file("agent_재고관리자.csv")
)
ordering_agent_df = load_ordering_agent_csv(
    find_csv_file("agent_발주담당자.csv")
)
budget_agent_df = load_budget_agent_csv(
    find_csv_file("agent_예산관리자.csv")
)
ORDER_RUN_AT = inventory_agent_df["snapshot_at"].max().to_pydatetime()

print(
    f"재고 {len(inventory_agent_df)}행 / 발주 {len(ordering_agent_df)}행 / "
    f"예산 후보 {len(budget_agent_df)}행"
)


## 9. 데이터검증결과 확인


In [ ]:
def load_validation_results(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    require_columns(df, ["결과"], path.name)
    df["결과"] = df["결과"].astype(str).str.upper().str.strip()
    return df


validation_df = load_validation_results(
    find_csv_file("데이터검증결과.csv")
)
validation_counts = {
    status: int((validation_df["결과"] == status).sum())
    for status in ["PASS", "WARN", "FAIL"]
}
print(
    f"데이터 검증 PASS={validation_counts['PASS']}, "
    f"WARN={validation_counts['WARN']}, FAIL={validation_counts['FAIL']}"
)
fail_rows = validation_df[validation_df["결과"] == "FAIL"]
warn_rows = validation_df[validation_df["결과"] == "WARN"]
if not fail_rows.empty:
    display(fail_rows)
    raise ValueError("데이터 검증 FAIL이 존재하여 Agent 실행을 중단합니다.")
if not warn_rows.empty:
    print("데이터 검증 WARN이 존재합니다.")
    display(warn_rows)


## 10. 데이터 간 교차검증


In [ ]:
def load_integrated_csv(path):
    if path is None:
        return None
    df = pd.read_csv(path, encoding="utf-8-sig")
    if "snapshot_at" in df:
        df["snapshot_at"] = pd.to_datetime(df["snapshot_at"], errors="raise")
    numeric_candidates = [
        "avg_daily_quantity_long", "available_qty",
        "recommended_order_qty_before_budget", "estimated_order_cost_krw",
    ]
    for column in numeric_candidates:
        if column in df:
            df[column] = pd.to_numeric(df[column], errors="raise")
    return df[df["store_id"].astype(str) == STORE_ID].copy()


integrated_df = load_integrated_csv(
    find_csv_file("06_통합발주분석.csv", required=False)
)


def cross_validate_agent_data():
    issues = []

    def add(area, item_id, detail):
        issues.append({
            "검증영역": area,
            "item_id": item_id,
            "결과": "REVIEW_REQUIRED",
            "상세": detail,
        })

    sales_ids = set(sales_agent_df["item_id"])
    inventory_ids = set(inventory_agent_df["item_id"])
    ordering_ids = set(ordering_agent_df["item_id"])
    budget_ids = set(budget_agent_df["item_id"])
    for item_id in sorted(sales_ids - inventory_ids):
        add("키 연결", item_id, "판매에는 있으나 재고에는 없음")
    for item_id in sorted(inventory_ids - ordering_ids):
        add("키 연결", item_id, "재고에는 있으나 발주에는 없음")
    order_candidates = set(
        ordering_agent_df.loc[
            ordering_agent_df["recommended_order_qty_before_budget_ea"] > 0,
            "item_id",
        ]
    )
    for item_id in sorted(budget_ids - order_candidates):
        add("키 연결", item_id, "예산 후보가 발주 후보에 없음")

    frames = {
        "판매": sales_agent_df,
        "재고": inventory_agent_df,
        "발주": ordering_agent_df,
        "예산": budget_agent_df,
    }
    store_sets = {
        name: set(df["store_id"].astype(str)) for name, df in frames.items()
    }
    if any(values != {STORE_ID} for values in store_sets.values()):
        add("매장", "*", f"store_id 불일치: {store_sets}")

    base_names = sales_agent_df.set_index("item_id")["item_name"]
    for name, frame in frames.items():
        names = frame.set_index("item_id")["item_name"]
        for item_id in sorted(set(base_names.index) & set(names.index)):
            if base_names[item_id] != names[item_id]:
                add("상품명", item_id, f"판매={base_names[item_id]}, {name}={names[item_id]}")

    def compare_numeric(left_df, right_df, column, left_name, right_name):
        merged = left_df[["item_id", column]].merge(
            right_df[["item_id", column]],
            on="item_id",
            suffixes=("_left", "_right"),
        )
        for _, row in merged.iterrows():
            difference = abs(float(row[f"{column}_left"]) - float(row[f"{column}_right"]))
            if difference > NUMERIC_TOLERANCE:
                add(
                    "수치 일치", row["item_id"],
                    f"{column}: {left_name}={row[f'{column}_left']}, "
                    f"{right_name}={row[f'{column}_right']}",
                )

    compare_numeric(
        sales_agent_df, inventory_agent_df,
        "avg_daily_quantity_long", "판매", "재고",
    )
    compare_numeric(
        inventory_agent_df, ordering_agent_df,
        "available_qty", "재고", "발주",
    )
    compare_numeric(
        ordering_agent_df, budget_agent_df,
        "recommended_order_qty_before_budget", "발주", "예산",
    )
    compare_numeric(
        ordering_agent_df, budget_agent_df,
        "estimated_order_cost_krw", "발주", "예산",
    )

    if integrated_df is not None:
        compare_numeric(
            inventory_agent_df, integrated_df,
            "available_qty", "재고", "통합",
        )
        compare_numeric(
            ordering_agent_df, integrated_df,
            "recommended_order_qty_before_budget", "발주", "통합",
        )
        compare_numeric(
            ordering_agent_df, integrated_df,
            "estimated_order_cost_krw", "발주", "통합",
        )
    return pd.DataFrame(
        issues,
        columns=["검증영역", "item_id", "결과", "상세"],
    )


cross_validation_df = cross_validate_agent_data()
if cross_validation_df.empty:
    print("Agent 간 키·상품명·핵심 수치 교차검증 PASS")
else:
    print(f"교차검증 REVIEW_REQUIRED: {len(cross_validation_df)}건")
    display(cross_validation_df)


## 11. 판매·재고·발주 Observation 생성


In [ ]:
# 모든 분류·수량·건수는 Python에서 확정하며 Agent는 이를 다시 계산하지 않습니다.
HIGH_VARIABILITY_CV = 0.6
q33 = sales_agent_df["avg_daily_quantity_long"].quantile(0.33)
q67 = sales_agent_df["avg_daily_quantity_long"].quantile(0.67)
sales_agent_df["demand_level"] = np.select(
    [
        sales_agent_df["avg_daily_quantity_long"] >= q67,
        sales_agent_df["avg_daily_quantity_long"] <= q33,
    ],
    ["HIGH", "LOW"],
    default="MEDIUM",
)
sales_agent_df["trend_status"] = np.select(
    [
        sales_agent_df["sales_trend_qty_per_day_long"] > 0.1,
        sales_agent_df["sales_trend_qty_per_day_long"] < -0.1,
    ],
    ["UP", "DOWN"],
    default="STABLE",
)
sales_agent_df["variability_level"] = np.where(
    sales_agent_df["cv_daily_quantity_long"] >= HIGH_VARIABILITY_CV,
    "HIGH",
    "NORMAL",
)


def validate_sales_rows(df):
    issues = {}
    for index, row in df.iterrows():
        validation_errors = []
        if row["avg_daily_quantity_short"] < 0:
            validation_errors.append("단기 평균 판매량 음수")
        if row["avg_daily_quantity_long"] < 0:
            validation_errors.append("장기 평균 판매량 음수")
        if row["std_daily_quantity_long"] < 0:
            validation_errors.append("판매 표준편차 음수")
        if row["cv_daily_quantity_long"] < 0:
            validation_errors.append("판매 변동계수 음수")
        if row["demand_level"] not in {"HIGH", "MEDIUM", "LOW"}:
            validation_errors.append("판매 수준 분류 오류")
        if row["trend_status"] not in {"UP", "DOWN", "STABLE"}:
            validation_errors.append("판매 추세 분류 오류")
        if row["variability_level"] not in {"HIGH", "NORMAL"}:
            validation_errors.append("판매 변동성 분류 오류")
        issues[index] = {
            "validation_errors": validation_errors,
            "review_flags": [],
        }
    return issues


def validate_inventory_rows(df):
    issues = {}
    for index, row in df.iterrows():
        validation_errors = []
        review_flags = []
        if row["available_qty"] < 0:
            validation_errors.append("available_qty 음수")
        if not math.isclose(
            row["inventory_position_qty"],
            row["available_qty"] + row["inbound_qty"],
            abs_tol=NUMERIC_TOLERANCE,
        ):
            validation_errors.append("inventory_position_qty 계산 불일치")
        if row["stockout_gap_qty"] < 0:
            validation_errors.append("stockout_gap_qty 음수")
        if row["waste_risk_qty"] < 0:
            validation_errors.append("waste_risk_qty 음수")
        if (
            pd.notna(row["nearest_removal_date"])
            and pd.notna(row["nearest_expiry_date"])
            and row["nearest_removal_date"] > row["nearest_expiry_date"]
        ):
            validation_errors.append("제거일이 소비기한보다 늦음")
        if row["waste_risk_yn"]:
            review_flags.append("FEFO 및 소비기한 확인 필요")
        issues[index] = {
            "validation_errors": validation_errors,
            "review_flags": review_flags,
        }
    return issues


def validate_ordering_rows(df):
    issues = {}
    for index, row in df.iterrows():
        validation_errors = []
        review_flags = []
        qty = row["recommended_order_qty_before_budget_ea"]
        if qty < 0:
            validation_errors.append("발주수량 음수")
        if row["estimated_order_cost_krw"] < 0:
            validation_errors.append("발주금액 음수")
        if qty > 0 and qty + NUMERIC_TOLERANCE < row["min_order_qty"]:
            validation_errors.append("MOQ 미충족")
        multiple = row["order_multiple_qty"]
        if qty > 0 and multiple > 0 and not math.isclose(
            qty / multiple, round(qty / multiple), abs_tol=NUMERIC_TOLERANCE
        ):
            validation_errors.append("발주 배수 불일치")
        if pd.notna(row["max_order_qty"]) and qty > row["max_order_qty"] + NUMERIC_TOLERANCE:
            validation_errors.append("최대 발주량 초과")
        expected_uom = qty / row["units_per_order_uom"]
        if not math.isclose(
            row["recommended_order_qty_before_budget_order_uom"],
            expected_uom,
            abs_tol=NUMERIC_TOLERANCE,
        ):
            validation_errors.append("발주단위 환산 불일치")
        expected_cost = qty * row["unit_cost_krw"]
        if not math.isclose(
            row["estimated_order_cost_krw"],
            expected_cost,
            abs_tol=NUMERIC_TOLERANCE,
        ):
            validation_errors.append("발주금액 계산 불일치")
        if row["net_required_qty"] <= 0 and qty > 0:
            validation_errors.append("필요수량 0 이하인데 발주 존재")
        if row["order_cap_applied_yn"]:
            review_flags.append("최대 발주량 또는 소비기한 상한 적용")
        issues[index] = {
            "validation_errors": validation_errors,
            "review_flags": review_flags,
        }
    return issues


sales_row_issues = validate_sales_rows(sales_agent_df)
inventory_row_issues = validate_inventory_rows(inventory_agent_df)
ordering_row_issues = validate_ordering_rows(ordering_agent_df)

sales_summary = {
    "total_sku_count": int(len(sales_agent_df)),
    "high_demand_sku_count": int((sales_agent_df["demand_level"] == "HIGH").sum()),
    "medium_demand_sku_count": int((sales_agent_df["demand_level"] == "MEDIUM").sum()),
    "low_demand_sku_count": int((sales_agent_df["demand_level"] == "LOW").sum()),
    "uptrend_sku_count": int((sales_agent_df["trend_status"] == "UP").sum()),
    "downtrend_sku_count": int((sales_agent_df["trend_status"] == "DOWN").sum()),
    "stable_sku_count": int((sales_agent_df["trend_status"] == "STABLE").sum()),
    "high_variability_sku_count": int((sales_agent_df["variability_level"] == "HIGH").sum()),
}
inventory_summary = {
    "total_sku_count": int(len(inventory_agent_df)),
    "stockout_risk_sku_count": int(inventory_agent_df["stockout_risk_yn"].fillna(False).astype(bool).sum()),
    "stockout_warning_sku_count": int(inventory_agent_df["stockout_warning_yn"].fillna(False).astype(bool).sum()),
    "waste_risk_sku_count": int(inventory_agent_df["waste_risk_yn"].fillna(False).astype(bool).sum()),
    "combined_risk_sku_count": int((
        inventory_agent_df["stockout_risk_yn"].fillna(False).astype(bool)
        & inventory_agent_df["waste_risk_yn"].fillna(False).astype(bool)
    ).sum()),
    "review_required_sku_count": int(sum(
        bool(value["validation_errors"] or value["review_flags"])
        for value in inventory_row_issues.values()
    )),
}
order_qty_column = "recommended_order_qty_before_budget_ea"
ordering_summary = {
    "total_sku_count": int(len(ordering_agent_df)),
    "order_candidate_sku_count": int((ordering_agent_df[order_qty_column] > 0).sum()),
    "zero_order_sku_count": int((ordering_agent_df[order_qty_column] <= 0).sum()),
    "capped_order_sku_count": int(ordering_agent_df["order_cap_applied_yn"].fillna(False).astype(bool).sum()),
    "review_required_sku_count": int(sum(
        bool(value["validation_errors"] or value["review_flags"])
        for value in ordering_row_issues.values()
    )),
    "estimated_order_cost_krw": int(round(ordering_agent_df["estimated_order_cost_krw"].sum())),
}


def make_observation(
    source_file,
    input_mode,
    df,
    columns,
    row_issues=None,
    summary=None,
):
    records = json_records(df, columns)
    all_errors = []
    all_flags = []
    for position, (index, item) in enumerate(df.iterrows()):
        issue = (row_issues or {}).get(index, {
            "validation_errors": [],
            "review_flags": [],
        })
        if isinstance(issue, list):  # 이전 형식도 안전하게 수용
            issue = {"validation_errors": issue, "review_flags": []}
        errors = list(issue.get("validation_errors", []))
        flags = list(issue.get("review_flags", []))
        status = "BLOCKED" if errors else ("REVIEW_REQUIRED" if flags else "OK")
        records[position].update({
            "validation_errors": errors,
            "review_flags": flags,
            "validation_results": errors + flags,
            "status": status,
        })
        all_errors.extend(f"{item['item_id']}: {message}" for message in errors)
        all_flags.extend(f"{item['item_id']}: {message}" for message in flags)
    result = {
        "metadata": {
            "store_id": STORE_ID,
            "source_file": source_file,
            "input_mode": input_mode,
            "row_count": len(records),
            "generated_at": datetime.now(timezone.utc).isoformat(),
        },
        "quality_summary": {
            "status": "FAIL" if all_errors else ("REVIEW_REQUIRED" if all_flags else "PASS"),
            "validation_errors": all_errors,
            "review_flags": all_flags,
            "issues": all_errors + all_flags,
        },
        "records": records,
    }
    if summary is not None:
        result["summary"] = summary
    return result


sales_columns = [
    "as_of_date", "store_id", "item_id", "item_name", "category_l1",
    "is_perishable", "avg_daily_quantity_short", "avg_daily_quantity_long",
    "std_daily_quantity_long", "cv_daily_quantity_long",
    "recent_to_long_ratio", "sales_trend_qty_per_day_long",
    "weekday_avg_quantity", "weekend_avg_quantity", "weekend_uplift_pct",
    "top_sales_weekday", "top_sales_weekday_avg_quantity",
    "promotion_quantity_share_pct", "zero_sales_days_long",
    "demand_level", "trend_status", "variability_level",
]
inventory_columns = [
    "snapshot_at", "store_id", "item_id", "item_name", "category_l1",
    "on_hand_qty", "reserved_qty", "inbound_qty", "available_qty",
    "inventory_position_qty", "inventory_cover_days", "lead_time_days",
    "lead_time_demand_qty", "stockout_gap_qty", "stockout_risk_yn",
    "stockout_warning_yn", "lot_count", "nearest_expiry_date",
    "nearest_removal_date", "removal_within_short_horizon_qty",
    "removal_within_medium_horizon_qty", "removal_within_long_horizon_qty",
    "waste_risk_qty", "waste_risk_yn",
]
ordering_columns = [
    "snapshot_at", "store_id", "item_id", "supplier_id", "item_name",
    "supplier_name", "target_stock_qty", "net_required_qty",
    "min_order_qty", "max_order_qty", "order_multiple_qty",
    "recommended_order_qty_before_budget_ea",
    "recommended_order_qty_before_budget_order_uom", "unit_cost_krw",
    "estimated_order_cost_krw", "order_cap_applied_yn",
]

sales_observation = make_observation(
    summary_path.name if SALES_INPUT_MODE == "SUMMARY" else daily_path.name,
    SALES_INPUT_MODE, sales_agent_df, sales_columns,
    row_issues=sales_row_issues, summary=sales_summary,
)
inventory_observation = make_observation(
    "agent_재고관리자.csv", None, inventory_agent_df, inventory_columns,
    row_issues=inventory_row_issues, summary=inventory_summary,
)
ordering_observation = make_observation(
    "agent_발주담당자.csv", None, ordering_agent_df, ordering_columns,
    row_issues=ordering_row_issues, summary=ordering_summary,
)

print("Python Observation summary")
display(pd.DataFrame([
    {"Agent": "sales_analyst", **sales_summary},
    {"Agent": "inventory_manager", **inventory_summary},
    {"Agent": "ordering_specialist", **ordering_summary},
]))


## 12. 예산 우선순위 조정


In [ ]:
def apply_budget_priority_adjustment(df):
    result = df.copy()
    pre_adjustment_cost_krw = float(result["estimated_order_cost_krw"].sum())
    limits = result["budget_limit_krw"].dropna().unique()
    if len(limits) != 1:
        raise ValueError(f"예산 한도가 하나가 아닙니다: {limits}")
    budget_limit_krw = float(limits[0])
    result = result.sort_values(
        [
            "stockout_risk_yn", "stockout_warning_yn", "waste_risk_yn",
            "inventory_cover_days", "service_level_target_pct", "item_id",
        ],
        ascending=[False, False, True, True, False, True],
    ).reset_index(drop=True)
    remaining = budget_limit_krw
    final_quantities = []
    for _, row in result.iterrows():
        candidate = float(row["recommended_order_qty_before_budget"])
        cost = float(row["unit_cost_krw"])
        multiple = float(row["order_multiple_qty"])
        minimum = float(row["min_order_qty"])
        maximum = float(row["max_order_qty"]) if pd.notna(row["max_order_qty"]) else candidate
        candidate = min(candidate, maximum)
        if pre_adjustment_cost_krw <= budget_limit_krw or candidate * cost <= remaining:
            final_qty = candidate
        else:
            affordable = math.floor((remaining / cost) / multiple) * multiple if cost > 0 and multiple > 0 else 0
            final_qty = min(candidate, maximum, affordable)
            if 0 < final_qty < minimum:
                final_qty = 0
        final_quantities.append(final_qty)
        remaining -= final_qty * cost
    result["final_order_qty_ea"] = final_quantities
    result["final_order_qty_order_uom"] = result["final_order_qty_ea"] / result["units_per_order_uom"]
    result["final_order_cost_krw"] = result["final_order_qty_ea"] * result["unit_cost_krw"]
    result["reduction_qty_ea"] = result["recommended_order_qty_before_budget"] - result["final_order_qty_ea"]
    result["reduction_cost_krw"] = result["estimated_order_cost_krw"] - result["final_order_cost_krw"]
    result["budget_adjusted_yn"] = result["reduction_qty_ea"] > NUMERIC_TOLERANCE
    result["adjustment_reason"] = np.where(
        result["budget_adjusted_yn"],
        "예산 초과로 우선순위 기반 감액",
        "예산 범위 내 후보 수량 유지",
    )
    result["final_status"] = np.where(result["budget_adjusted_yn"], "ADJUSTED", "OK")
    return result


budget_result_df = apply_budget_priority_adjustment(budget_agent_df)
final_order_df = budget_result_df
budget_limit_krw = float(budget_result_df["budget_limit_krw"].iloc[0])
pre_adjustment_cost_krw = float(budget_result_df["estimated_order_cost_krw"].sum())
final_order_cost_krw = float(budget_result_df["final_order_cost_krw"].sum())
total_reduction_cost_krw = pre_adjustment_cost_krw - final_order_cost_krw
remaining_budget_krw = budget_limit_krw - final_order_cost_krw
budget_status = "PASS" if final_order_cost_krw <= budget_limit_krw + NUMERIC_TOLERANCE else "BLOCKED"
budget_allowed_item_ids = set(budget_result_df["item_id"].astype(str).tolist())
budget_summary = {
    "budget_limit_krw": int(round(budget_limit_krw)),
    "pre_adjustment_cost_krw": int(round(pre_adjustment_cost_krw)),
    "final_order_cost_krw": int(round(final_order_cost_krw)),
    "total_reduction_cost_krw": int(round(total_reduction_cost_krw)),
    "remaining_budget_krw": int(round(remaining_budget_krw)),
    "candidate_sku_count": int(len(budget_result_df)),
    "final_order_sku_count": int((budget_result_df["final_order_qty_ea"] > 0).sum()),
    "adjusted_sku_count": int(budget_result_df["budget_adjusted_yn"].fillna(False).astype(bool).sum()),
    "zeroed_sku_count": int((
        (budget_result_df["recommended_order_qty_before_budget"] > 0)
        & (budget_result_df["final_order_qty_ea"] == 0)
    ).sum()),
    "budget_status": budget_status,
}


def validate_budget_result(df, summary):
    checks = []
    for _, row in df.iterrows():
        qty = row["final_order_qty_ea"]
        candidate = row["recommended_order_qty_before_budget"]
        maximum = row["max_order_qty"]
        multiple = row["order_multiple_qty"]
        minimum = row["min_order_qty"]
        checks.extend([
            (f"{row['item_id']} 수량 비음수", qty >= 0),
            (f"{row['item_id']} 후보 이하", qty <= candidate + NUMERIC_TOLERANCE),
            (f"{row['item_id']} 최대량 이하", pd.isna(maximum) or qty <= maximum + NUMERIC_TOLERANCE),
            (f"{row['item_id']} MOQ", qty == 0 or qty + NUMERIC_TOLERANCE >= minimum),
            (f"{row['item_id']} 배수", qty == 0 or math.isclose(qty / multiple, round(qty / multiple), abs_tol=NUMERIC_TOLERANCE)),
            (f"{row['item_id']} 금액", math.isclose(row["final_order_cost_krw"], qty * row["unit_cost_krw"], abs_tol=NUMERIC_TOLERANCE)),
        ])
    checks.extend([
        ("전체 예산", summary["final_order_cost_krw"] <= summary["budget_limit_krw"]),
        ("남은 예산", summary["remaining_budget_krw"] >= 0),
    ])
    return pd.DataFrame(checks, columns=["검증항목", "통과"])


budget_validation_df = validate_budget_result(budget_result_df, budget_summary)
if not budget_validation_df["통과"].all():
    display(budget_validation_df[~budget_validation_df["통과"]])
    raise ValueError("예산 우선순위 조정 검증 실패")
print("예산 우선순위 조정 검증 PASS")
display(pd.DataFrame([budget_summary]))


## 13. 예산 Observation 생성


In [ ]:
budget_columns = [
    "snapshot_at", "store_id", "item_id", "supplier_id", "item_name",
    "recommended_order_qty_before_budget", "final_order_qty_ea",
    "final_order_qty_order_uom", "unit_cost_krw", "estimated_order_cost_krw",
    "final_order_cost_krw", "reduction_qty_ea", "reduction_cost_krw",
    "budget_adjusted_yn", "adjustment_reason", "final_status",
]
budget_observation = make_observation(
    "agent_예산관리자.csv",
    "PYTHON_PRIORITY_ADJUSTMENT",
    budget_result_df,
    budget_columns,
    summary=budget_summary,
)


def observation_task_context(observation):
    """Task에는 원본 records 없이 metadata와 quality_summary만 전달합니다."""
    return compact_json({
        "metadata": observation["metadata"],
        "quality_summary": observation["quality_summary"],
    })


demo_inputs = {
    "store_id": STORE_ID,
    "order_run_at": ORDER_RUN_AT.isoformat(),
    "sales_input_mode": SALES_INPUT_MODE,
    "sales_context_json": observation_task_context(sales_observation),
    "inventory_context_json": observation_task_context(inventory_observation),
    "ordering_context_json": observation_task_context(ordering_observation),
    "budget_context_json": observation_task_context(budget_observation),
}
print("Observation 생성 PASS")
for name, observation in [
    ("sales", sales_observation),
    ("inventory", inventory_observation),
    ("ordering", ordering_observation),
    ("budget", budget_observation),
]:
    print(name, observation["metadata"]["row_count"], observation["quality_summary"]["status"], observation["summary"])


## CoT·SC·ReAct 실행 도구

- **CoT:** 각 Agent가 데이터 품질 → 수치 대조 → 위험/제약 판정 → 행동 제안 순서로 내부 검토합니다. 내부 사고 전문은 출력하지 않고 검증 결과와 근거만 출력합니다.
- **ReAct:** Agent가 자신의 Observation 조회 도구를 실제 호출(Action)하고 반환값(Observation)을 근거로 판단합니다.
- **SC:** 각 Agent의 SC 도구가 동일한 검토 문제를 독립적으로 3회 LLM 호출하고, 본 Agent가 2개 이상 일치하는 결론만 채택합니다. 합의되지 않으면 `REVIEW_REQUIRED`로 표시합니다.

SC는 호출 수와 토큰 비용을 증가시키므로 `SELF_CONSISTENCY_PATHS=3`으로 고정하고, 수량·금액은 변경하지 않은 채 해석의 안정성만 검토합니다.


In [ ]:
from collections import Counter
import re

SELF_CONSISTENCY_PATHS = 3
tool_call_counts = Counter()
tool_call_log = []
sc_consensus_results = {}
sc_usage = Counter()


def lookup_observation(observation, item_id):
    if str(item_id).upper() == "ALL":
        return compact_json(observation)
    selected = [
        record for record in observation["records"]
        if str(record.get("item_id")) == str(item_id)
    ]
    return compact_json({
        "metadata": observation["metadata"],
        "quality_summary": observation["quality_summary"],
        "summary": observation.get("summary", {}),
        "records": selected,
    })


def parse_json_object(value):
    if hasattr(value, "model_dump"):
        return value.model_dump()
    if isinstance(value, dict):
        return value
    text = str(value).strip()
    fenced = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, re.S)
    if fenced:
        text = fenced.group(1)
    else:
        start, end = text.find("{"), text.rfind("}")
        if start >= 0 and end > start:
            text = text[start:end + 1]
    return json.loads(text)


def parse_sc_json(value):
    parsed = parse_json_object(value)
    required = {"judgment", "status", "evidence_fields", "review_reasons"}
    if set(parsed) != required:
        raise ValueError(f"SC 키 불일치: {sorted(parsed)}")
    allowed = {"PASS", "REVIEW_REQUIRED", "BLOCKED"}
    if parsed["judgment"] not in allowed:
        raise ValueError("SC judgment 허용값 오류")
    if parsed["status"] not in {"OK", "REVIEW_REQUIRED", "BLOCKED"}:
        raise ValueError("SC status 허용값 오류")
    if not isinstance(parsed["evidence_fields"], list) or not isinstance(parsed["review_reasons"], list):
        raise ValueError("SC list 필드 오류")
    return parsed


def aggregate_sc_results(sc_results):
    parsed_results, parse_errors = [], []
    for result in sc_results:
        try:
            parsed_results.append(parse_sc_json(result))
        except Exception as error:
            parse_errors.append(str(error))
    if len(parsed_results) < 2:
        return {
            "consensus_status": "REVIEW_REQUIRED",
            "consensus_judgment": "REVIEW_REQUIRED",
            "agreement_count": len(parsed_results),
            "path_count": len(sc_results),
            "parse_errors": parse_errors,
            "common_evidence_fields": [],
            "common_review_reasons": [],
            "results": parsed_results,
        }
    judgment_counts = Counter(result["judgment"] for result in parsed_results)
    top_judgment, agreement_count = judgment_counts.most_common(1)[0]
    # 파싱 실패나 3경로 불일치는 보수적으로 REVIEW_REQUIRED입니다.
    unanimous = agreement_count == len(sc_results) and not parse_errors
    consensus_judgment = top_judgment if unanimous else "REVIEW_REQUIRED"
    evidence_sets = [set(map(str, result["evidence_fields"])) for result in parsed_results]
    reason_sets = [set(map(str, result["review_reasons"])) for result in parsed_results]
    return {
        "consensus_status": "OK" if unanimous else "REVIEW_REQUIRED",
        "consensus_judgment": consensus_judgment,
        "agreement_count": agreement_count,
        "path_count": len(sc_results),
        "parse_errors": parse_errors,
        "common_evidence_fields": sorted(set.intersection(*evidence_sets)) if evidence_sets else [],
        "common_review_reasons": sorted(set.intersection(*reason_sets)) if reason_sets else [],
        "results": parsed_results,
    }


def estimate_tokens(text):
    try:
        from litellm import token_counter
        return int(token_counter(model=normalize_openai_model_name(model_name), text=str(text)))
    except Exception:
        return max(1, len(str(text)) // 4)


def run_sc_paths(agent_key, agent_llm, domain, observation, focus):
    evidence = compact_json(observation)
    base_prompt = (
        f"당신은 {domain} 교차검증자입니다. 독립적으로 검토하세요.\n"
        f"검토 초점: {focus}\n검증된 Observation: {evidence}\n"
        '반드시 다음 네 키만 가진 JSON을 출력하세요: '
        '{"judgment":"PASS | REVIEW_REQUIRED | BLOCKED",'
        '"status":"OK | REVIEW_REQUIRED | BLOCKED",'
        '"evidence_fields":[],"review_reasons":[]}\n'
        "Observation 숫자와 item_id를 변경·재계산·추가하지 말고 내부 사고과정은 출력하지 마세요."
    )
    raw_results = []
    for path_number in range(1, SELF_CONSISTENCY_PATHS + 1):
        prompt = base_prompt + f"\n독립 검토 경로 번호: {path_number}"
        response = agent_llm.call(prompt)
        raw_results.append(response)
        sc_usage["successful_requests"] += 1
        sc_usage["prompt_tokens"] += estimate_tokens(prompt)
        sc_usage["completion_tokens"] += estimate_tokens(response)
    sc_usage["total_tokens"] = sc_usage["prompt_tokens"] + sc_usage["completion_tokens"]
    consensus = aggregate_sc_results(raw_results)
    sc_consensus_results[agent_key] = consensus
    return compact_json(consensus)


def record_tool_call(name, argument):
    tool_call_counts[name] += 1
    tool_call_log.append({"tool": name, "argument": str(argument)})


@tool("sales_observation_lookup", max_usage_count=2)
def sales_observation_lookup(item_id: str = "ALL") -> str:
    """판매 Observation을 item_id 또는 ALL로 조회합니다. 판단 전 반드시 호출하세요."""
    record_tool_call("sales_observation_lookup", item_id)
    return lookup_observation(sales_observation, item_id)


@tool("sales_self_consistency_review", max_usage_count=1)
def sales_self_consistency_review(focus: str) -> str:
    """판매 해석의 3개 독립 경로와 Python 합의 결과를 반환합니다."""
    record_tool_call("sales_self_consistency_review", focus)
    return run_sc_paths("sales_analyst", sales_llm, "판매분석", sales_observation, focus)


@tool("inventory_observation_lookup", max_usage_count=2)
def inventory_observation_lookup(item_id: str = "ALL") -> str:
    """재고 Observation을 item_id 또는 ALL로 조회합니다. 판단 전 반드시 호출하세요."""
    record_tool_call("inventory_observation_lookup", item_id)
    return lookup_observation(inventory_observation, item_id)


@tool("inventory_self_consistency_review", max_usage_count=1)
def inventory_self_consistency_review(focus: str) -> str:
    """재고 해석의 3개 독립 경로와 Python 합의 결과를 반환합니다."""
    record_tool_call("inventory_self_consistency_review", focus)
    return run_sc_paths("inventory_manager", inventory_llm, "재고·소비기한", inventory_observation, focus)


@tool("ordering_observation_lookup", max_usage_count=2)
def ordering_observation_lookup(item_id: str = "ALL") -> str:
    """발주 Observation을 item_id 또는 ALL로 조회합니다. 판단 전 반드시 호출하세요."""
    record_tool_call("ordering_observation_lookup", item_id)
    return lookup_observation(ordering_observation, item_id)


@tool("ordering_self_consistency_review", max_usage_count=1)
def ordering_self_consistency_review(focus: str) -> str:
    """발주 해석의 3개 독립 경로와 Python 합의 결과를 반환합니다."""
    record_tool_call("ordering_self_consistency_review", focus)
    return run_sc_paths("ordering_specialist", ordering_llm, "발주 제약", ordering_observation, focus)


@tool("budget_observation_lookup", max_usage_count=2)
def budget_observation_lookup(item_id: str = "ALL") -> str:
    """예산 Observation을 item_id 또는 ALL로 조회합니다. 판단 전 반드시 호출하세요."""
    record_tool_call("budget_observation_lookup", item_id)
    return lookup_observation(budget_observation, item_id)


@tool("budget_self_consistency_review", max_usage_count=1)
def budget_self_consistency_review(focus: str) -> str:
    """예산 해석의 3개 독립 경로와 Python 합의 결과를 반환합니다."""
    record_tool_call("budget_self_consistency_review", focus)
    return run_sc_paths("budget_manager", budget_llm, "예산 우선순위 조정", budget_observation, focus)


## 14. 4개 Agent 정의


In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field

Status = Literal["OK", "REVIEW_REQUIRED", "BLOCKED"]


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class SalesSummary(StrictModel):
    total_sku_count: int
    high_demand_sku_count: int
    medium_demand_sku_count: int
    low_demand_sku_count: int
    uptrend_sku_count: int
    downtrend_sku_count: int
    stable_sku_count: int
    high_variability_sku_count: int


class SalesDetail(StrictModel):
    item_id: str
    item_name: str
    demand_level: Literal["HIGH", "MEDIUM", "LOW"]
    trend_status: Literal["UP", "DOWN", "STABLE"]
    variability_level: Literal["HIGH", "NORMAL"]
    status: Status
    reason: str
    evidence_fields: list[str]


class SalesAnalysisOutput(StrictModel):
    summary: SalesSummary
    details: list[SalesDetail]


class InventorySummary(StrictModel):
    total_sku_count: int
    stockout_risk_sku_count: int
    stockout_warning_sku_count: int
    waste_risk_sku_count: int
    combined_risk_sku_count: int
    review_required_sku_count: int


class InventoryDetail(StrictModel):
    item_id: str
    item_name: str
    on_hand_qty: float
    available_qty: float
    inventory_position_qty: float
    inventory_cover_days: float
    stockout_gap_qty: float
    stockout_risk_yn: bool
    stockout_warning_yn: bool
    waste_risk_qty: float
    waste_risk_yn: bool
    status: Status
    reason: str
    evidence_fields: list[str]


class InventoryAssessmentOutput(StrictModel):
    summary: InventorySummary
    details: list[InventoryDetail]


class OrderingSummary(StrictModel):
    total_sku_count: int
    order_candidate_sku_count: int
    zero_order_sku_count: int
    capped_order_sku_count: int
    review_required_sku_count: int
    estimated_order_cost_krw: int


class OrderingDetail(StrictModel):
    item_id: str
    item_name: str
    supplier_id: str
    supplier_name: str
    target_stock_qty: float
    net_required_qty: float
    min_order_qty: float
    max_order_qty: float
    order_multiple_qty: float
    recommended_order_qty_before_budget_ea: float
    recommended_order_qty_before_budget_order_uom: float
    unit_cost_krw: float
    estimated_order_cost_krw: float
    order_cap_applied_yn: bool
    status: Status
    reason: str
    evidence_fields: list[str]


class OrderingDraftOutput(StrictModel):
    summary: OrderingSummary
    details: list[OrderingDetail]


class BudgetSummary(StrictModel):
    budget_limit_krw: int
    pre_adjustment_cost_krw: int
    final_order_cost_krw: int
    total_reduction_cost_krw: int
    remaining_budget_krw: int
    candidate_sku_count: int
    final_order_sku_count: int
    adjusted_sku_count: int
    zeroed_sku_count: int
    budget_status: Literal["PASS", "BLOCKED"]


class BudgetDetail(StrictModel):
    item_id: str
    item_name: str
    supplier_id: str
    recommended_order_qty_before_budget: float
    final_order_qty_ea: float
    final_order_qty_order_uom: float
    unit_cost_krw: float
    estimated_order_cost_krw: float
    final_order_cost_krw: float
    reduction_qty_ea: float
    reduction_cost_krw: float
    budget_adjusted_yn: bool
    status: Status
    reason: str
    evidence_fields: list[str]


class BudgetAdjustmentOutput(StrictModel):
    summary: BudgetSummary
    # Python 후보 수와 동일한 9개를 누락·추가 없이 요구합니다.
    details: list[BudgetDetail] = Field(min_length=9, max_length=9)


sales_analyst = Agent(
    role="프랜차이즈 판매분석가",
    goal="Python 판매 summary와 Observation records를 그대로 설명한다.",
    backstory="판매 Observation만 사용한다. ReAct와 SC 3경로를 호출하고 CoT는 내부에서만 수행한다.",
    llm=sales_llm, tools=[sales_observation_lookup, sales_self_consistency_review],
    allow_delegation=False, max_iter=4, verbose=True,
)
inventory_manager = Agent(
    role="재고 및 소비기한 관리자",
    goal="Python 재고 summary와 Observation records를 그대로 설명한다.",
    backstory="재고 Observation만 details 근거로 사용한다. ReAct와 SC 3경로를 호출하고 CoT는 내부에서만 수행한다.",
    llm=inventory_llm, tools=[inventory_observation_lookup, inventory_self_consistency_review],
    allow_delegation=False, max_iter=4, verbose=True,
)
ordering_specialist = Agent(
    role="가맹점 발주담당자",
    goal="Python 발주 summary와 발주 관련 Observation records만 그대로 설명한다.",
    backstory="재고 보고서를 만들지 않는다. ReAct와 SC 3경로를 호출하고 CoT는 내부에서만 수행한다.",
    llm=ordering_llm, tools=[ordering_observation_lookup, ordering_self_consistency_review],
    allow_delegation=False, max_iter=4, verbose=True,
)
budget_manager = Agent(
    role="발주 예산관리자",
    goal="Python 예산 summary와 9개 후보 records를 누락 없이 그대로 설명한다.",
    backstory="예산 Observation 밖 SKU를 출력하지 않는다. ReAct와 SC 3경로를 호출하고 CoT는 내부에서만 수행한다.",
    llm=budget_llm, tools=[budget_observation_lookup, budget_self_consistency_review],
    allow_delegation=False, max_iter=4, verbose=True,
)


## 15. 4개 Task 정의


In [ ]:
REASONING_OUTPUT_RULE = (
    "반드시 1) 전용 Observation Lookup Tool을 item_id='ALL'로 호출하고, "
    "2) 전용 Self-Consistency Tool을 정확히 1회 호출하며, "
    "3) Tool이 반환한 Python consensus 결과를 확인한다. "
    "Tool 결과에 없는 숫자나 item_id는 사용하지 않는다. "
    "summary는 Observation summary를 복사하고 records를 직접 세거나 재집계하지 않는다. "
    "details item_id는 해당 역할 Observation records 안에서만 선택한다. "
    "수량·금액·bool을 다시 계산하거나 수정하지 않는다. "
    "내부 CoT와 SC 원문은 출력하지 않고 지정된 JSON 스키마만 출력한다."
)

analyze_sales = Task(
    description=(
        "판매 안내(원본 records 제외):\n{sales_context_json}\n"
        + REASONING_OUTPUT_RULE
        + " sales_observation.summary 숫자를 그대로 사용한다. HIGH/MEDIUM/LOW와 "
        "UP/DOWN/STABLE 및 변동성 건수를 records에서 다시 세지 않는다. "
        "details item_id는 sales_observation.records에 존재하는 값만 사용한다."
    ),
    expected_output="SalesAnalysisOutput 스키마의 JSON",
    output_pydantic=SalesAnalysisOutput,
    agent=sales_analyst,
    tools=[sales_observation_lookup, sales_self_consistency_review],
)
assess_inventory = Task(
    description=(
        "재고 안내(원본 records 제외):\n{inventory_context_json}\n"
        + REASONING_OUTPUT_RULE
        + " 위험 건수는 inventory_observation.summary를 그대로 쓴다. 판매 Context는 참고만 하고 "
        "판매 Context의 상품·수치를 재고 details에 추가하지 않는다."
    ),
    expected_output="InventoryAssessmentOutput 스키마의 JSON",
    output_pydantic=InventoryAssessmentOutput,
    agent=inventory_manager,
    context=[analyze_sales],
    tools=[inventory_observation_lookup, inventory_self_consistency_review],
)
draft_order = Task(
    description=(
        "발주 안내(원본 records 제외):\n{ordering_context_json}\n"
        + REASONING_OUTPUT_RULE
        + " summary는 ordering_observation.summary를 그대로 쓴다. details는 발주 관련 필드만 사용한다. "
        "stockout_count, waste_count 같은 재고 summary나 on_hand_qty 중심 보고서를 만들지 않는다. "
        "발주 후보 수량과 금액을 직접 계산하지 않는다. Observation 밖 item_id를 추가하지 않는다."
    ),
    expected_output="OrderingDraftOutput 스키마의 JSON",
    output_pydantic=OrderingDraftOutput,
    agent=ordering_specialist,
    context=[analyze_sales, assess_inventory],
    tools=[ordering_observation_lookup, ordering_self_consistency_review],
)
adjust_budget = Task(
    description=(
        "예산 안내(원본 records 제외):\n{budget_context_json}\n"
        + REASONING_OUTPUT_RULE
        + " details는 budget_observation.records의 후보 전체를 누락 없이 출력한다. "
        "다른 Context에만 있는 상품을 출력하거나 9개 후보를 다른 상품으로 대체하지 않는다. "
        "candidate_sku_count와 details 개수를 일치시키고 최종 수량·금액을 재계산하지 않는다."
    ),
    expected_output="BudgetAdjustmentOutput 스키마의 JSON",
    output_pydantic=BudgetAdjustmentOutput,
    agent=budget_manager,
    context=[assess_inventory, draft_order],
    tools=[budget_observation_lookup, budget_self_consistency_review],
)


## 16. Crew 정의


In [ ]:
crew = Crew(
    agents=[
        sales_analyst,
        inventory_manager,
        ordering_specialist,
        budget_manager,
    ],
    tasks=[
        analyze_sales,
        assess_inventory,
        draft_order,
        adjust_budget,
    ],
    process=Process.sequential,
    verbose=True,
)


## 17. Crew 실행


In [ ]:
print(f"선택 판매 모드: {SALES_INPUT_MODE}")
print(f"판매/재고/발주/예산 행 수: {len(sales_agent_df)}/{len(inventory_agent_df)}/{len(ordering_agent_df)}/{len(budget_result_df)}")
crew_result = None
crew_error = None
try:
    crew_result = await crew.kickoff_async(inputs=demo_inputs)
except Exception as exc:
    crew_error = exc
    print(f"Crew 실행 실패: {type(exc).__name__}: {exc}")


def task_parsed_output(task):
    if task.output is None:
        raise ValueError("Task 출력 없음")
    if getattr(task.output, "pydantic", None) is not None:
        return task.output.pydantic.model_dump()
    if getattr(task.output, "json_dict", None) is not None:
        return task.output.json_dict
    return parse_json_object(task.output.raw)


def comparable_equal(expected, actual):
    if isinstance(expected, bool):
        return isinstance(actual, bool) and expected == actual
    if isinstance(expected, (int, float)) and isinstance(actual, (int, float)):
        return math.isclose(float(expected), float(actual), abs_tol=NUMERIC_TOLERANCE)
    return expected == actual


def validate_agent_output(
    agent_name,
    parsed_output,
    observation,
    expected_summary,
    required_detail_fields,
    require_all_records=False,
):
    rows = []
    def add(item, passed, expected, actual, basis):
        rows.append({
            "Agent": agent_name, "검증 항목": item,
            "결과": "PASS" if passed else "FAIL",
            "기대값": expected, "실제값": actual, "판단 근거": basis,
        })
    try:
        parsed = parsed_output.model_dump() if hasattr(parsed_output, "model_dump") else (
            parsed_output if isinstance(parsed_output, dict) else parse_json_object(parsed_output)
        )
        add("JSON/Pydantic 파싱", True, "객체", type(parsed).__name__, "정상 파싱")
    except Exception as error:
        add("JSON/Pydantic 파싱", False, "객체", str(error), "파싱 실패")
        return rows
    summary = parsed.get("summary")
    details = parsed.get("details")
    add("summary 필수 필드", isinstance(summary, dict) and set(summary) == set(expected_summary),
        sorted(expected_summary), sorted(summary) if isinstance(summary, dict) else summary, "역할별 summary 스키마")
    if isinstance(summary, dict):
        for key, expected in expected_summary.items():
            actual = summary.get(key)
            add(f"summary.{key}", comparable_equal(expected, actual), expected, actual, "Python summary 기준")
    add("details list", isinstance(details, list), "list", type(details).__name__, "구조 검증")
    if not isinstance(details, list):
        return rows
    allowed_records = {str(record["item_id"]): record for record in observation["records"]}
    ids = [str(detail.get("item_id")) for detail in details if isinstance(detail, dict)]
    invalid_ids = sorted(set(ids) - set(allowed_records))
    duplicates = sorted(item_id for item_id, count in Counter(ids).items() if count > 1)
    missing_ids = sorted(set(allowed_records) - set(ids))
    add("details item_id 유효성", not invalid_ids, "Observation 부분집합", invalid_ids, "허용되지 않은 SKU 생성 금지")
    add("details 중복 없음", not duplicates, "중복 없음", duplicates, "item_id 유일성")
    if require_all_records:
        add("후보 SKU 누락 없음", not missing_ids and len(details) == len(allowed_records),
            len(allowed_records), len(details), f"누락={missing_ids}")
    numeric_mismatches = []
    status_errors = []
    field_errors = []
    for detail in details:
        if not isinstance(detail, dict):
            field_errors.append("detail이 객체가 아님")
            continue
        missing = sorted(set(required_detail_fields) - set(detail))
        if missing:
            field_errors.append(f"{detail.get('item_id')}: {missing}")
        status = detail.get("status")
        if status not in {"OK", "REVIEW_REQUIRED", "BLOCKED"}:
            status_errors.append(f"{detail.get('item_id')}={status}")
        record = allowed_records.get(str(detail.get("item_id")))
        if record:
            for field, actual in detail.items():
                if field in record and isinstance(record[field], (int, float, bool)) and not comparable_equal(record[field], actual):
                    numeric_mismatches.append(f"{detail.get('item_id')}.{field}: {record[field]} != {actual}")
    add("필수 detail 필드", not field_errors, sorted(required_detail_fields), field_errors, "역할별 스키마")
    add("수량·금액 Observation 일치", not numeric_mismatches, "모두 일치", numeric_mismatches[:20], "Python record 기준")
    add("status 허용값", not status_errors, "OK/REVIEW_REQUIRED/BLOCKED", status_errors, "Literal 검증")
    return rows


agent_specs = {
    "sales_analyst": (analyze_sales, sales_observation, sales_summary, set(SalesDetail.model_fields), False, SalesAnalysisOutput),
    "inventory_manager": (assess_inventory, inventory_observation, inventory_summary, set(InventoryDetail.model_fields), False, InventoryAssessmentOutput),
    "ordering_specialist": (draft_order, ordering_observation, ordering_summary, set(OrderingDetail.model_fields), False, OrderingDraftOutput),
    "budget_manager": (adjust_budget, budget_observation, budget_summary, set(BudgetDetail.model_fields), True, BudgetAdjustmentOutput),
}
agent_validation_rows = []
pydantic_validation = {}
parsed_agent_outputs = {}
for agent_name, (task, observation, expected, fields, require_all, model) in agent_specs.items():
    try:
        parsed = task_parsed_output(task)
        model.model_validate(parsed)
        pydantic_validation[agent_name] = True
        parsed_agent_outputs[agent_name] = parsed
    except Exception as error:
        pydantic_validation[agent_name] = False
        parsed = {"parse_error": str(error)}
    agent_validation_rows.extend(validate_agent_output(
        agent_name, parsed, observation, expected, fields, require_all
    ))

agent_validation_df = pd.DataFrame(agent_validation_rows)
if crew_error is not None:
    agent_validation_df = pd.concat([agent_validation_df, pd.DataFrame([{
        "Agent": "Crew", "검증 항목": "Crew 실행", "결과": "FAIL",
        "기대값": "성공", "실제값": str(crew_error), "판단 근거": "실행 예외",
    }])], ignore_index=True)
required_tools = {
    "sales_observation_lookup", "inventory_observation_lookup",
    "ordering_observation_lookup", "budget_observation_lookup",
    "sales_self_consistency_review", "inventory_self_consistency_review",
    "ordering_self_consistency_review", "budget_self_consistency_review",
}
react_tool_validation = {name: tool_call_counts[name] >= 1 for name in sorted(required_tools)}
sc_python_validation = {
    name: (
        result.get("path_count") == 3
        and result.get("agreement_count", 0) >= 2
        and not result.get("parse_errors")
    )
    for name, result in sc_consensus_results.items()
}
all_agent_rows_pass = not agent_validation_df.empty and (agent_validation_df["결과"] == "PASS").all()
final_agent_validation_status = (
    "PASS" if all_agent_rows_pass
    and all(pydantic_validation.values())
    and all(react_tool_validation.values())
    and len(sc_python_validation) == 4
    and all(sc_python_validation.values())
    else "FAIL"
)
print("=== Agent 출력 사후 검증 ===")
display(agent_validation_df)
print("ReAct Tool:", react_tool_validation)
print("SC Python 합의:", sc_consensus_results)
print("Pydantic:", pydantic_validation)
print("final_agent_validation_status =", final_agent_validation_status)


## 18. OpenAI API 토큰·비용 분석

이 영역은 점주용 최종 발주 검토 결과와 분리된 **기술·비용 분석**입니다.

### 분석 요구사항 반영

| 사용자 요구사항 | 반영 내용 |
|---|---|
| Input·Output 토큰 분석 | 입력, 캐시 입력, 출력, 전체 토큰과 요청당 평균을 표시 |
| 실제 달러 비용 계산 | OpenAI 공식 모델 단가를 적용해 입력·캐시·출력·총비용 계산 |
| 비용 절감 방안 | 현재 구조의 비용 원인을 진단하고 실행 가능한 절감 방안 제시 |
| 사진의 수업 기준 반영 | Tokenization·Embedding 차이, 토큰 수와 비용·성능 관계, 서브워드 토큰화 장점 설명 |

### 수업 개념 연결

- **Tokenization:** 입력 문장과 compact JSON을 모델이 처리할 수 있는 서브워드 토큰으로 나누는 과정입니다. API 비용은 이 입력 토큰과 생성된 출력 토큰 수를 기준으로 계산됩니다.
- **Embedding:** 각 토큰을 문맥 계산에 사용할 수 있는 연속 벡터로 바꾸는 내부 표현 과정입니다. 이 Crew는 별도 Embeddings API를 호출하지 않으므로 임베딩 API 비용은 없지만, 토큰 수가 많아질수록 이후 Self-Attention의 처리량과 지연이 커집니다.
- **서브워드 토큰화의 장점:** 처음 보는 상품명·한글·영문 컬럼명도 부분 단위로 표현할 수 있어 미등록 단어 문제를 줄입니다. 반면 동일 의미라도 표현과 언어에 따라 토큰 수가 달라질 수 있으므로 compact JSON과 짧은 필드 선택이 중요합니다.

단가는 노트북 기본 모델 `gpt-4o-mini`의 공식 가격인 100만 토큰당 일반 입력 $0.15, 캐시 입력 $0.075, 출력 $0.60을 사용합니다. 모델을 변경하면 가격표도 공식 OpenAI 가격에 맞게 갱신해야 합니다.


In [ ]:
# OpenAI 공식 GPT-4o mini 가격(USD / 1M text tokens)
# https://developers.openai.com/api/docs/models/gpt-4o-mini
OPENAI_PRICING_USD_PER_1M = {
    "gpt-4o-mini": {
        "input": 0.15,
        "cached_input": 0.075,
        "output": 0.60,
    },
}

TOKEN_USAGE_OUTPUT_PATH = OUTPUT_DIR / "order_balance_api_token_usage.csv"
TOKEN_ANALYSIS_OUTPUT_PATH = OUTPUT_DIR / "order_balance_api_token_analysis.md"


def normalize_openai_model_name(name):
    return str(name).split("/")[-1]


def calculate_openai_token_cost(usage, model):
    """CrewAI 실측 usage에 OpenAI의 토큰 단가를 적용합니다."""
    normalized_model = normalize_openai_model_name(model)
    if normalized_model not in OPENAI_PRICING_USD_PER_1M:
        raise ValueError(
            f"{normalized_model}의 단가가 등록되지 않았습니다. "
            "OpenAI 공식 모델 페이지에서 최신 단가를 확인해 가격표를 갱신하세요."
        )

    rates = OPENAI_PRICING_USD_PER_1M[normalized_model]
    prompt_tokens = int(usage.get("prompt_tokens", 0))
    cached_prompt_tokens = min(
        int(usage.get("cached_prompt_tokens", 0)),
        prompt_tokens,
    )
    uncached_prompt_tokens = max(0, prompt_tokens - cached_prompt_tokens)
    completion_tokens = int(usage.get("completion_tokens", 0))
    successful_requests = int(usage.get("successful_requests", 0))

    uncached_input_cost = (
        uncached_prompt_tokens / 1_000_000 * rates["input"]
    )
    cached_input_cost = (
        cached_prompt_tokens / 1_000_000 * rates["cached_input"]
    )
    output_cost = completion_tokens / 1_000_000 * rates["output"]
    total_cost = uncached_input_cost + cached_input_cost + output_cost

    return {
        "model": normalized_model,
        "prompt_tokens": prompt_tokens,
        "uncached_prompt_tokens": uncached_prompt_tokens,
        "cached_prompt_tokens": cached_prompt_tokens,
        "completion_tokens": completion_tokens,
        "reasoning_tokens": int(usage.get("reasoning_tokens", 0)),
        "total_tokens": int(
            usage.get(
                "total_tokens",
                prompt_tokens + completion_tokens,
            )
        ),
        "successful_requests": successful_requests,
        "input_rate_usd_per_1m": rates["input"],
        "cached_input_rate_usd_per_1m": rates["cached_input"],
        "output_rate_usd_per_1m": rates["output"],
        "uncached_input_cost_usd": uncached_input_cost,
        "cached_input_cost_usd": cached_input_cost,
        "output_cost_usd": output_cost,
        "total_cost_usd": total_cost,
    }


def build_token_cost_analysis(cost):
    """실측 비용 진단, 절감 시나리오, 사용자 지시 반영 내용을 작성합니다."""
    request_count = max(cost["successful_requests"], 1)
    input_tokens = cost["prompt_tokens"]
    output_tokens = cost["completion_tokens"]
    billable_tokens = input_tokens + output_tokens
    input_share = input_tokens / billable_tokens * 100 if billable_tokens else 0
    output_share = output_tokens / billable_tokens * 100 if billable_tokens else 0

    # 개선 효과를 비교하기 위한 보수적인 예시 시나리오
    reduced_uncached_input = round(cost["uncached_prompt_tokens"] * 0.70)
    reduced_cached_input = round(cost["cached_prompt_tokens"] * 0.70)
    reduced_output = round(output_tokens * 0.80)
    reduced_cost = (
        reduced_uncached_input / 1_000_000 * cost["input_rate_usd_per_1m"]
        + reduced_cached_input / 1_000_000 * cost["cached_input_rate_usd_per_1m"]
        + reduced_output / 1_000_000 * cost["output_rate_usd_per_1m"]
    )
    savings = cost["total_cost_usd"] - reduced_cost
    savings_pct = (
        savings / cost["total_cost_usd"] * 100
        if cost["total_cost_usd"] > 0
        else 0
    )

    analysis_lines = [
        "# Order Balance OpenAI API 토큰·비용 분석",
        "",
        "## 1. 사용자 지시사항 반영",
        "- Input·Output 토큰을 분리해 실측했습니다.",
        "- OpenAI 공식 단가로 실제 달러 비용을 계산했습니다.",
        "- 비용 절감 방안과 정량 절감 시나리오를 제시했습니다.",
        "- 점주용 최종 발주 검토 결과와 분리된 기술 분석으로 작성했습니다.",
        "- Tokenization·Embedding 수업 평가 기준과 연결했습니다.",
        "",
        "## 2. 실측 토큰 사용량",
        f"- 모델: {cost['model']}",
        f"- 일반 입력 토큰: {cost['uncached_prompt_tokens']:,}",
        f"- 캐시 입력 토큰: {cost['cached_prompt_tokens']:,}",
        f"- 전체 입력 토큰: {input_tokens:,}",
        f"- 출력 토큰: {output_tokens:,}",
        f"- 전체 토큰: {cost['total_tokens']:,}",
        f"- 성공 API 요청 수: {cost['successful_requests']:,}",
        f"- 요청당 평균 입력 토큰: {input_tokens / request_count:,.1f}",
        f"- 요청당 평균 출력 토큰: {output_tokens / request_count:,.1f}",
        "",
        "## 3. 실제 달러 비용",
        f"- 일반 입력 비용: ${cost['uncached_input_cost_usd']:.8f}",
        f"- 캐시 입력 비용: ${cost['cached_input_cost_usd']:.8f}",
        f"- 출력 비용: ${cost['output_cost_usd']:.8f}",
        f"- 총 API 비용: ${cost['total_cost_usd']:.8f}",
        (
            "- 계산식: (일반 입력 토큰 ÷ 1,000,000 × 입력 단가) + "
            "(캐시 입력 토큰 ÷ 1,000,000 × 캐시 단가) + "
            "(출력 토큰 ÷ 1,000,000 × 출력 단가)"
        ),
        "",
        "## 4. 비용·성능 해석",
        f"- 전체 청구 대상 토큰 중 입력 비중은 {input_share:.1f}%, 출력 비중은 {output_share:.1f}%입니다.",
        (
            "- 입력 비중이 높으면 4개 Observation과 이전 Task context의 반복 전달이 주요 비용 원인입니다."
            if input_share >= output_share
            else "- 출력 비중이 높으면 36개 SKU 전체를 반복 설명하는 긴 Agent 출력이 주요 비용 원인입니다."
        ),
        "- 토큰 수 증가는 비용뿐 아니라 컨텍스트 처리량과 응답 지연도 증가시킵니다.",
        "- Embedding은 토큰을 벡터로 바꾸는 내부 단계이며, 본 Crew에서는 별도 Embeddings API 비용이 발생하지 않습니다.",
        "",
        "## 5. 토큰 비용 절감 방안",
        "1. 각 Agent Observation에서 해당 역할이 실제 사용하는 컬럼만 유지합니다.",
        "2. 36개 전체 SKU는 요약 필드만 전달하고, 위험·발주 후보 SKU에만 상세 필드를 전달합니다.",
        "3. 다음 Task에 이전 Agent의 전체 JSON을 넘기지 말고 key_findings와 REVIEW_REQUIRED 항목만 context로 전달합니다.",
        "4. final Task의 details는 발주 후보 9개와 위험 품목으로 제한하고 정상 0발주 품목은 집계 요약으로 대체합니다.",
        "5. 프롬프트의 반복 제약 문장을 공통 prefix로 고정해 Prompt Caching 활용 가능성을 높입니다.",
        "6. expected_output의 중복 설명을 줄이고 JSON 키 목록은 한 번만 선언합니다.",
        "7. 품목별 설명을 1문장 또는 정해진 글자 수로 제한해 출력 토큰을 제어합니다.",
        "8. 개발 중에는 1개 매장·소수 SKU 샘플로 검증하고 최종 실행에서만 전체 데이터를 사용합니다.",
        "",
        "## 6. 정량 절감 시나리오",
        "- 가정: 입력 토큰 30% 감소, 출력 토큰 20% 감소",
        f"- 개선 전 비용: ${cost['total_cost_usd']:.8f}",
        f"- 개선 후 예상 비용: ${reduced_cost:.8f}",
        f"- 예상 절감액: ${savings:.8f}",
        f"- 예상 절감률: {savings_pct:.1f}%",
        "",
        "## 7. 수업 개념 연결",
        "- Tokenization은 텍스트·JSON을 서브워드 토큰으로 분해하는 과정이며 토큰 수가 API 비용의 직접 기준입니다.",
        "- Embedding은 각 토큰을 문맥 계산용 벡터로 바꾸는 단계로 Tokenization과 역할이 다릅니다.",
        "- 서브워드 방식은 한글 상품명이나 미등록 단어도 부분 단위로 표현할 수 있지만 표현 방식에 따라 토큰 수가 달라집니다.",
        "- 4개 Agent의 순차 호출은 매번 토큰화→임베딩→Self-Attention→생성을 반복하므로 비용과 시간이 누적됩니다.",
        "",
        "가격 출처: https://developers.openai.com/api/docs/models/gpt-4o-mini",
    ]
    return "\n".join(analysis_lines), {
        "input_share_pct": input_share,
        "output_share_pct": output_share,
        "projected_reduced_cost_usd": reduced_cost,
        "projected_savings_usd": savings,
        "projected_savings_pct": savings_pct,
    }



if crew_result is not None:
    crew_usage = crew_result.token_usage.model_dump()
    # CrewAI 집계가 직접 agent_llm.call로 실행한 SC 12회를 포함하지 않으면 별도 누적합니다.
    crew_requests = int(crew_usage.get("successful_requests", 0))
    sc_included_in_crew_usage = crew_requests >= 20
    usage = dict(crew_usage)
    if not sc_included_in_crew_usage:
        for key in ["successful_requests", "prompt_tokens", "completion_tokens", "total_tokens"]:
            usage[key] = int(usage.get(key, 0)) + int(sc_usage.get(key, 0))
    usage["cached_prompt_tokens"] = int(usage.get("cached_prompt_tokens", 0))
    base_agent_requests = max(
        0,
        crew_requests - int(sc_usage["successful_requests"])
        if sc_included_in_crew_usage else crew_requests,
    )
    token_cost = calculate_openai_token_cost(usage, model_name)
    token_analysis_markdown, token_savings = build_token_cost_analysis(token_cost)
    token_analysis_markdown += (
        "\n\n## 8. 전체 요청 구성 근거\n"
        f"- 기본 Agent/ReAct 호출: {base_agent_requests}회\n"
        f"- SC 3경로 추가 호출: {sc_usage['successful_requests']}회(4 Agent × 3경로)\n"
        "- Tool 기반 ReAct는 Agent별 Observation 조회, SC Tool 선택, 최종 구조화 출력 단계로 호출 수를 증가시킵니다.\n"
        f"- SC 사용량의 Crew 집계 포함 여부: {'포함' if sc_included_in_crew_usage else '미포함 — 별도 합산'}\n"
        f"- 최종 전체 요청 수: {token_cost['successful_requests']}회"
    )
    api_token_usage_df = pd.DataFrame([{
        **token_cost, **token_savings,
        "base_agent_requests": base_agent_requests,
        "sc_additional_requests": int(sc_usage["successful_requests"]),
        "sc_usage_already_in_crew": sc_included_in_crew_usage,
        "agent_count": len(crew.agents),
        "reasoning_method": "CoT + Python 3-path SC + ReAct",
        "sc_paths_per_agent": SELF_CONSISTENCY_PATHS,
        "task_count": len(crew.tasks),
        "measured_at": datetime.now(timezone.utc).isoformat(),
    }])
    api_token_usage_df.to_csv(TOKEN_USAGE_OUTPUT_PATH, index=False, encoding="utf-8-sig")
    TOKEN_ANALYSIS_OUTPUT_PATH.write_text(token_analysis_markdown, encoding="utf-8")
    print("=== OpenAI API 토큰·비용 분석 ===")
    display(api_token_usage_df)
    display(Markdown(token_analysis_markdown))
else:
    api_token_usage_df = pd.DataFrame([{"status": "UNAVAILABLE", "reason": str(crew_error)}])
    token_analysis_markdown = "# OpenAI API 토큰·비용 분석\n\nCrew 실행 결과가 없어 계산할 수 없습니다."
    display(api_token_usage_df)


## 19. 점주용 Markdown 결과


In [ ]:
def render_store_order_report():
    inventory_final = inventory_agent_df.merge(
        final_order_df[["item_id", "final_order_qty_ea"]],
        on="item_id",
        how="left",
    )
    stockout_df = inventory_final[inventory_final["stockout_risk_yn"]].copy()
    waste_df = inventory_agent_df[inventory_agent_df["waste_risk_yn"]].copy()
    adjusted_df = final_order_df[final_order_df["budget_adjusted_yn"]].copy()

    review_items = []
    for index, issue in inventory_row_issues.items():
        review_items.extend(
            f"- {inventory_agent_df.loc[index, 'item_id']}: {message}"
            for message in issue["validation_errors"] + issue["review_flags"]
        )
    for index, issue in ordering_row_issues.items():
        review_items.extend(
            f"- {ordering_agent_df.loc[index, 'item_id']}: {message}"
            for message in issue["validation_errors"] + issue["review_flags"]
        )
    combined = inventory_agent_df[
        inventory_agent_df["stockout_risk_yn"]
        & inventory_agent_df["waste_risk_yn"]
    ]
    review_items.extend(
        f"- {row['item_id']}: 품절·폐기 복합 위험 확인"
        for _, row in combined.iterrows()
    )
    review_items.extend(
        f"- {row['item_id']}: 예산 감액 수량 확인"
        for _, row in adjusted_df.iterrows()
    )

    lines = [
        "# Order Balance 최종 발주 검토 결과",
        "",
        ("Agent 출력 검증 불일치로 수치 정보는 Python 검증 결과를 사용했습니다."
         if final_agent_validation_status != "PASS"
         else "Agent 출력 사후 검증 PASS — 수치 정보는 Python 검증 DataFrame 기준입니다."),
        "",
        "## 1. 실행 정보",
        (
            f"{sales_agent_df['item_id'].nunique()}개 SKU를 분석해 "
            f"{len(budget_agent_df)}개 발주 후보를 검토했고, "
            f"{budget_summary['final_order_sku_count']}개가 최종 발주 대상입니다."
        ),
        f"- 매장: {STORE_ID}",
        f"- 실행 기준일: {ORDER_RUN_AT.isoformat()}",
        f"- 판매 입력 모드: {SALES_INPUT_MODE}",
        f"- 판매 분석 SKU 수: {sales_agent_df['item_id'].nunique()}",
        f"- 발주 후보 SKU 수: {len(budget_agent_df)}",
        f"- 최종 발주 SKU 수: {budget_summary['final_order_sku_count']}",
        "",
        "## 2. 예산 요약",
        (
            f"최종 발주금액은 "
            f"{budget_summary['final_order_cost_krw']:,.0f}원으로 예산의 "
            f"{budget_summary['final_order_cost_krw'] / budget_summary['budget_limit_krw'] * 100:.1f}%를 "
            f"사용하며, {budget_summary['remaining_budget_krw']:,.0f}원이 남습니다."
        ),
        f"- 예산 한도: {budget_summary['budget_limit_krw']:,.0f}원",
        f"- 조정 전 발주금액: {budget_summary['pre_adjustment_cost_krw']:,.0f}원",
        f"- 최종 발주금액: {budget_summary['final_order_cost_krw']:,.0f}원",
        f"- 감액 금액: {budget_summary['total_reduction_cost_krw']:,.0f}원",
        f"- 남은 예산: {budget_summary['remaining_budget_krw']:,.0f}원",
        f"- 예산 상태: {budget_summary['budget_status']}",
        "",
        "## 3. 최종 발주 상세",
    ]

    if adjusted_df.empty:
        lines.append(
            f"{len(final_order_df)}개 발주 후보를 모두 확인한 결과, "
            "예산·MOQ·발주 배수·최대량 조건을 충족하여 수량 조정 없이 그대로 발주 가능합니다."
        )
    else:
        lines.append(
            f"{len(final_order_df)}개 후보 중 {len(adjusted_df)}개 품목의 수량이 "
            f"조정되었고, 최종 발주금액은 {budget_summary['final_order_cost_krw']:,.0f}원입니다."
        )
    lines += [
        "|상품ID|상품명|공급사|예산 전 수량|최종 수량|발주단위 수량|단가|최종 금액|조정|사유|",
        "|---|---|---|---:|---:|---:|---:|---:|---|---|",
    ]
    for _, row in final_order_df.iterrows():
        lines.append(
            f"|{row['item_id']}|{row['item_name']}|{row['supplier_id']}|"
            f"{row['recommended_order_qty_before_budget']:.2f}|{row['final_order_qty_ea']:.2f}|"
            f"{row['final_order_qty_order_uom']:.2f}|{row['unit_cost_krw']:,.0f}|"
            f"{row['final_order_cost_krw']:,.0f}|{'Y' if row['budget_adjusted_yn'] else 'N'}|"
            f"{row['adjustment_reason']}|"
        )

    lines += ["", "## 4. 품절 위험"]
    if stockout_df.empty:
        lines.append(
            "재고커버일·리드타임·부족수량을 확인한 결과, 품절 위험 상품은 없습니다."
        )
        lines.append("- 품절 위험 상세 내역 없음")
    else:
        lines.append(
            f"품절 위험 상품은 {len(stockout_df)}개이며, "
            f"리드타임 기준 총 부족수량은 {stockout_df['stockout_gap_qty'].sum():.2f}개입니다."
        )
        lines += [
            "|상품|재고커버일|리드타임|부족수량|최종 발주수량|",
            "|---|---:|---:|---:|---:|",
        ]
        for _, row in stockout_df.iterrows():
            lines.append(
                f"|{row['item_name']}|{row['inventory_cover_days']:.2f}|"
                f"{row['lead_time_days']:.0f}|{row['stockout_gap_qty']:.2f}|"
                f"{row['final_order_qty_ea'] if pd.notna(row['final_order_qty_ea']) else 0:.2f}|"
            )

    lines += ["", "## 5. 폐기 위험 및 FEFO"]
    if waste_df.empty:
        lines.append(
            "제거일·소비기한·재고량을 확인한 결과, 폐기 위험 상품은 없습니다."
        )
        lines.append("- 폐기 위험 및 FEFO 상세 내역 없음")
    else:
        lines.append(
            f"폐기 위험 상품은 {len(waste_df)}개이며, "
            f"총 폐기 위험수량은 {waste_df['waste_risk_qty'].sum():.2f}개이므로 FEFO 확인이 필요합니다."
        )
        lines += [
            "|상품|가장 가까운 소비기한|폐기 위험수량|확인 행동|",
            "|---|---|---:|---|",
        ]
        for _, row in waste_df.iterrows():
            lines.append(
                f"|{row['item_name']}|{row['nearest_expiry_date']}|"
                f"{row['waste_risk_qty']:.2f}|FEFO 진열·할인·폐기 예정 확인|"
            )

    lines += ["", "## 6. 예산 조정 내역"]
    if adjusted_df.empty:
        lines.append(
            f"조정 전 발주금액 {budget_summary['pre_adjustment_cost_krw']:,.0f}원이 "
            f"예산 한도 {budget_summary['budget_limit_krw']:,.0f}원 이내이므로 감액 또는 제외된 품목이 없습니다."
        )
        lines.append("- 예산 조정 내역 없음 — 모든 후보 수량을 그대로 유지했습니다.")
    else:
        lines.append(
            f"예산 우선순위 조정으로 {len(adjusted_df)}개 품목에서 "
            f"총 {adjusted_df['reduction_cost_krw'].sum():,.0f}원을 감액했습니다."
        )
        lines += [
            "|상품|감액 전|감액 후|감액 금액|이유|",
            "|---|---:|---:|---:|---|",
        ]
        for _, row in adjusted_df.iterrows():
            lines.append(
                f"|{row['item_name']}|{row['recommended_order_qty_before_budget']:.2f}|"
                f"{row['final_order_qty_ea']:.2f}|{row['reduction_cost_krw']:,.0f}|"
                f"{row['adjustment_reason']}|"
            )

    lines += ["", "## 7. 점주 확인 필요 항목"]
    if review_items:
        lines.append(
            f"점주가 최종 승인 전에 확인해야 할 항목은 총 {len(review_items)}건입니다."
        )
        lines.extend(review_items)
    else:
        lines.append(
            "데이터 검증, 복합 위험, 예산 감액 기준에서 별도의 점주 확인 항목이 없습니다."
        )
        lines.append("- 별도 확인 항목 없음")

    lines += [
        "",
        "**본 결과는 자동 발주 전송 결과가 아니라 점주 검토용 발주 초안입니다.**",
    ]
    return "\n".join(lines)


store_order_report = render_store_order_report()
print("\n=== 최종 예산 요약 ===")
display(pd.DataFrame([budget_summary]))
display(Markdown(store_order_report))


## 20. 최종 CSV 및 PDF 저장


In [ ]:
FINAL_OUTPUT_COLUMNS = [
    "snapshot_at", "store_id", "item_id", "item_name", "supplier_id",
    "category_l1", "order_uom", "units_per_order_uom", "min_order_qty",
    "max_order_qty", "order_multiple_qty",
    "recommended_order_qty_before_budget", "final_order_qty_ea",
    "final_order_qty_order_uom", "unit_cost_krw",
    "estimated_order_cost_krw", "final_order_cost_krw",
    "reduction_qty_ea", "reduction_cost_krw", "stockout_risk_yn",
    "stockout_warning_yn", "waste_risk_yn", "budget_adjusted_yn",
    "adjustment_reason", "final_status",
]
FINAL_OUTPUT_PATH = OUTPUT_DIR / "order_balance_final_order.csv"
final_order_df[FINAL_OUTPUT_COLUMNS].to_csv(
    FINAL_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)
print(f"최종 CSV 저장 위치: {FINAL_OUTPUT_PATH}")


In [ ]:
import html
import re

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.platypus import (
    PageBreak,
    Paragraph,
    SimpleDocTemplate,
    Spacer,
    Table,
    TableStyle,
)


def find_korean_font():
    """macOS·Windows·Linux에서 사용 가능한 한글 TrueType 폰트를 찾습니다."""
    candidates = [
        Path("/System/Library/Fonts/Supplemental/Arial Unicode.ttf"),
        Path("/Library/Fonts/AppleGothic.ttf"),
        Path("C:/Windows/Fonts/malgun.ttf"),
        Path("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"),
        Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"),
    ]
    for font_path in candidates:
        if font_path.exists():
            return font_path
    raise FileNotFoundError(
        "한글 PDF 생성용 폰트를 찾지 못했습니다. Noto Sans CJK 또는 맑은 고딕을 설치하세요."
    )


def clean_markdown_text(text):
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r"`(.*?)`", r"\1", text)
    return html.escape(text)


def markdown_to_pdf(markdown_text, output_path):
    """점주용 Markdown 보고서를 표가 포함된 다중 페이지 PDF로 저장합니다."""
    font_name = "OrderBalanceKorean"
    if font_name not in pdfmetrics.getRegisteredFontNames():
        pdfmetrics.registerFont(TTFont(font_name, str(find_korean_font())))

    page_size = landscape(A4)
    document = SimpleDocTemplate(
        str(output_path),
        pagesize=page_size,
        rightMargin=12 * mm,
        leftMargin=12 * mm,
        topMargin=13 * mm,
        bottomMargin=14 * mm,
        title="Order Balance 최종 발주 검토 결과",
        author="Order Balance",
    )
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        "KoreanTitle",
        parent=styles["Title"],
        fontName=font_name,
        fontSize=19,
        leading=24,
        textColor=colors.HexColor("#17365D"),
        alignment=TA_CENTER,
        spaceAfter=10,
    )
    heading_style = ParagraphStyle(
        "KoreanHeading",
        parent=styles["Heading2"],
        fontName=font_name,
        fontSize=13,
        leading=17,
        textColor=colors.HexColor("#1F4E79"),
        spaceBefore=9,
        spaceAfter=5,
    )
    body_style = ParagraphStyle(
        "KoreanBody",
        parent=styles["BodyText"],
        fontName=font_name,
        fontSize=8.5,
        leading=12,
        textColor=colors.HexColor("#202124"),
        spaceAfter=3,
    )
    table_style = ParagraphStyle(
        "KoreanTable",
        parent=body_style,
        fontSize=6.2,
        leading=8,
        alignment=TA_CENTER,
    )

    story = []
    lines = markdown_text.splitlines()
    index = 0
    while index < len(lines):
        stripped = lines[index].strip()
        if not stripped:
            story.append(Spacer(1, 2 * mm))
            index += 1
            continue

        if stripped.startswith("|"):
            table_lines = []
            while index < len(lines) and lines[index].strip().startswith("|"):
                table_lines.append(lines[index].strip())
                index += 1
            parsed_rows = [
                [cell.strip() for cell in line.strip("|").split("|")]
                for line in table_lines
            ]
            parsed_rows = [
                row for row in parsed_rows
                if not all(re.fullmatch(r":?-{3,}:?", cell) for cell in row)
            ]
            if parsed_rows:
                column_count = max(len(row) for row in parsed_rows)
                normalized_rows = [
                    row + [""] * (column_count - len(row))
                    for row in parsed_rows
                ]
                table_data = [
                    [
                        Paragraph(clean_markdown_text(cell), table_style)
                        for cell in row
                    ]
                    for row in normalized_rows
                ]
                available_width = page_size[0] - 24 * mm
                table = Table(
                    table_data,
                    colWidths=[available_width / column_count] * column_count,
                    repeatRows=1,
                    hAlign="LEFT",
                )
                table.setStyle(TableStyle([
                    ("FONTNAME", (0, 0), (-1, -1), font_name),
                    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9EAF7")),
                    ("TEXTCOLOR", (0, 0), (-1, 0), colors.HexColor("#17365D")),
                    ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#A6A6A6")),
                    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                    ("LEFTPADDING", (0, 0), (-1, -1), 2),
                    ("RIGHTPADDING", (0, 0), (-1, -1), 2),
                    ("TOPPADDING", (0, 0), (-1, -1), 2),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
                    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [
                        colors.white,
                        colors.HexColor("#F7F9FC"),
                    ]),
                ]))
                story.extend([table, Spacer(1, 3 * mm)])
            continue

        if stripped.startswith("# "):
            story.append(Paragraph(clean_markdown_text(stripped[2:]), title_style))
        elif stripped.startswith("## "):
            story.append(Paragraph(clean_markdown_text(stripped[3:]), heading_style))
        elif stripped.startswith("- "):
            story.append(
                Paragraph("• " + clean_markdown_text(stripped[2:]), body_style)
            )
        else:
            story.append(Paragraph(clean_markdown_text(stripped), body_style))
        index += 1

    def add_page_number(canvas, document):
        canvas.saveState()
        canvas.setFont(font_name, 7)
        canvas.setFillColor(colors.HexColor("#777777"))
        canvas.drawCentredString(
            page_size[0] / 2,
            7 * mm,
            f"Order Balance · {document.page}",
        )
        canvas.restoreState()

    document.build(
        story,
        onFirstPage=add_page_number,
        onLaterPages=add_page_number,
    )
    return output_path


FINAL_PDF_PATH = OUTPUT_DIR / "order_balance_final_order_report.pdf"
markdown_to_pdf(store_order_report, FINAL_PDF_PATH)
print(f"최종 PDF 저장 위치: {FINAL_PDF_PATH}")


## 21. 통합 자동 검증


In [ ]:
agent_objects = [sales_analyst, inventory_manager, ordering_specialist, budget_manager]
task_objects = [analyze_sales, assess_inventory, draft_order, adjust_budget]
verification_rows = []


def verify(area, item, passed, basis, failure_status="FAIL"):
    verification_rows.append({
        "검증 영역": area, "검증 항목": item,
        "결과": "PASS" if bool(passed) else failure_status,
        "판단 근거": basis,
    })


# 1) 시스템 구조 검증
verify("system_structure_validation", "Agent 4개", len(agent_objects) == 4, [a.role for a in agent_objects])
verify("system_structure_validation", "Task 4개", len(task_objects) == 4, len(task_objects))
verify("system_structure_validation", "지정 순서", crew.agents == agent_objects and crew.tasks == task_objects, "판매→재고→발주→예산")
verify("system_structure_validation", "Process.sequential", crew.process == Process.sequential, str(crew.process))
verify("system_structure_validation", "SUMMARY/DAILY_DETAIL 비교", (sales_mode_comparison_df["결과"] == "PASS").all(), sales_mode_comparison_df["결과"].value_counts().to_dict())
verify("system_structure_validation", "Python 행 계산 검증", (
    not any(v["validation_errors"] for v in sales_row_issues.values())
    and not any(v["validation_errors"] for v in inventory_row_issues.values())
    and not any(v["validation_errors"] for v in ordering_row_issues.values())
    and budget_validation_df["통과"].all()
), "sales/inventory/ordering/budget")
verify("system_structure_validation", "Observation summary 포함", all(
    "summary" in value for value in [sales_observation, inventory_observation, ordering_observation, budget_observation]
), "4개 Observation")
verify("system_structure_validation", "CSV 생성", FINAL_OUTPUT_PATH.exists(), str(FINAL_OUTPUT_PATH))
verify("system_structure_validation", "PDF 생성", FINAL_PDF_PATH.exists(), str(FINAL_PDF_PATH))

# 2) Agent 출력 정확성
summary_agent_pass = {}
for agent_name in agent_specs:
    scoped = agent_validation_df[
        (agent_validation_df["Agent"] == agent_name)
        & agent_validation_df["검증 항목"].str.startswith("summary.")
    ]
    summary_agent_pass[agent_name] = not scoped.empty and (scoped["결과"] == "PASS").all()
verify("agent_output_validation", "판매 summary Python 일치", summary_agent_pass.get("sales_analyst"), sales_summary)
verify("agent_output_validation", "재고 summary Python 일치", summary_agent_pass.get("inventory_manager"), inventory_summary)
verify("agent_output_validation", "발주 summary Python 일치", summary_agent_pass.get("ordering_specialist"), ordering_summary)
verify("agent_output_validation", "예산 summary Python 일치", summary_agent_pass.get("budget_manager"), budget_summary)
for item in ["details item_id 유효성", "details 중복 없음"]:
    scoped = agent_validation_df[agent_validation_df["검증 항목"] == item]
    verify("agent_output_validation", f"Agent별 {item}", len(scoped) == 4 and (scoped["결과"] == "PASS").all(), scoped[["Agent", "결과"]].to_dict("records"))
verify("agent_output_validation", "발주담당자 출력 스키마 적합", pydantic_validation.get("ordering_specialist"), "OrderingDraftOutput extra=forbid")
verify("agent_output_validation", "예산 후보 SKU 외 생성 없음", set(
    detail["item_id"] for detail in parsed_agent_outputs.get("budget_manager", {}).get("details", [])
) <= budget_allowed_item_ids, sorted(budget_allowed_item_ids))
verify("agent_output_validation", "예산 후보 SKU 누락 여부", set(
    detail["item_id"] for detail in parsed_agent_outputs.get("budget_manager", {}).get("details", [])
) == budget_allowed_item_ids, sorted(budget_allowed_item_ids))
verify("agent_output_validation", "SC Python 다수결 성공", len(sc_python_validation) == 4 and all(sc_python_validation.values()), sc_python_validation)
verify("agent_output_validation", "ReAct Tool 호출", all(react_tool_validation.values()), react_tool_validation)
verify("agent_output_validation", "Pydantic Schema 검증 성공", all(pydantic_validation.values()), pydantic_validation)
verify("agent_output_validation", "Agent 출력 검증 전체 PASS", final_agent_validation_status == "PASS", final_agent_validation_status)

verification_df = pd.DataFrame(verification_rows)
system_structure_validation = (
    "PASS" if (verification_df.loc[
        verification_df["검증 영역"] == "system_structure_validation", "결과"
    ] == "PASS").all() else "FAIL"
)
agent_output_validation = (
    "PASS" if (verification_df.loc[
        verification_df["검증 영역"] == "agent_output_validation", "결과"
    ] == "PASS").all() else "FAIL"
)
overall_validation_status = (
    "PASS" if system_structure_validation == agent_output_validation == "PASS"
    else ("REVIEW_REQUIRED" if system_structure_validation == "PASS" else "FAIL")
)
display(verification_df)

summary_rows = [
    ["판매", "전체/HIGH/MEDIUM/LOW", f"{sales_summary['total_sku_count']}/{sales_summary['high_demand_sku_count']}/{sales_summary['medium_demand_sku_count']}/{sales_summary['low_demand_sku_count']}"],
    ["판매", "UP/DOWN/STABLE", f"{sales_summary['uptrend_sku_count']}/{sales_summary['downtrend_sku_count']}/{sales_summary['stable_sku_count']}"],
    ["재고", "품절/경고/폐기/복합", f"{inventory_summary['stockout_risk_sku_count']}/{inventory_summary['stockout_warning_sku_count']}/{inventory_summary['waste_risk_sku_count']}/{inventory_summary['combined_risk_sku_count']}"],
    ["발주", "전체/후보/0수량/상한", f"{ordering_summary['total_sku_count']}/{ordering_summary['order_candidate_sku_count']}/{ordering_summary['zero_order_sku_count']}/{ordering_summary['capped_order_sku_count']}"],
    ["발주", "조정 전 금액", ordering_summary["estimated_order_cost_krw"]],
    ["예산", "후보/최종/감액/0수량", f"{budget_summary['candidate_sku_count']}/{budget_summary['final_order_sku_count']}/{budget_summary['adjusted_sku_count']}/{budget_summary['zeroed_sku_count']}"],
    ["예산", "한도/최종금액/잔액", f"{budget_summary['budget_limit_krw']}/{budget_summary['final_order_cost_krw']}/{budget_summary['remaining_budget_krw']}"],
    ["SC", "Agent별 합의 수", {k: v.get("agreement_count") for k, v in sc_consensus_results.items()}],
    ["ReAct", "Agent별 Tool 호출", "PASS" if all(react_tool_validation.values()) else "FAIL"],
    ["Schema", "Agent별 Pydantic 검증", "PASS" if all(pydantic_validation.values()) else "FAIL"],
    ["Agent 출력", "Python 값 일치", final_agent_validation_status],
    ["파일", "CSV/PDF 생성", "PASS" if FINAL_OUTPUT_PATH.exists() and FINAL_PDF_PATH.exists() else "FAIL"],
    ["전체", "최종 검증", overall_validation_status],
]
final_summary_df = pd.DataFrame(summary_rows, columns=["영역", "항목", "결과"])
display(final_summary_df)
print("system_structure_validation =", system_structure_validation)
print("agent_output_validation =", agent_output_validation)
if overall_validation_status == "PASS":
    print("Order Balance 4-Agent 워크플로우 및 Agent 출력 통합 검증 PASS")
else:
    print(f"전체 통합 검증 {overall_validation_status}: Agent 오류를 전체 PASS로 처리하지 않았습니다.")


## 22. 수업 개념 연결

- CSV의 상품별 요약은 토큰화되어 LLM 입력으로 전달됩니다.
- 각 Agent는 Observation 내부 필드 관계를 Self-Attention으로 해석합니다.
- 4개 Agent가 순차 호출되므로 호출마다 Transformer 추론 비용과 실행 시간이 누적됩니다.
- 역할 분리는 판매·재고·발주·예산이라는 서로 다른 검토 관점과 설명 가능성을 높입니다.


## 23. 한계 및 개선 방향

1. **판매분석가:** 36행 SUMMARY 또는 6,480행 DAILY_DETAIL의 Python 집계 결과를 해석합니다.
2. **재고관리자:** 가용재고·품절·폐기·FEFO 위험을 검토합니다.
3. **발주담당자:** 목표재고·필요수량·MOQ·배수·최대량을 검증합니다.
4. **예산관리자:** Python의 예산 우선순위 조정 결과를 검증하고 최종 통합합니다.

Python은 계산과 제약 검증을, LLM은 해석과 설명을 담당합니다. 예산 조정은 수학적 최적해가 아닌 휴리스틱이며 날씨·행사·향후 프로모션 계획, 자동 발주 전송은 포함하지 않습니다. 최종 의사결정은 점주가 수행합니다.

향후에는 정수계획법 기반 최적화, 실제 발주 API, 점주 승인 워크플로우, 도구 호출형 ReAct, 판매 예측모델을 연동할 수 있습니다.
